# FB-LARA Kaggle Single-File Notebook

This notebook is self-contained. It writes the embedded `zakotfs_novelty` package, configs, and tests into the Kaggle working directory, then runs the FB-LARA novelty pipeline.

Upload to Kaggle:
- `full_cnn_best.pt` or `train_history_full.zip`
- optionally `train_full.json` / `val_full.json` plus their `.npy` memmaps from the 480k baseline dataset

If the baseline train/val manifests are present, adapter dataset generation reuses them directly and skips the slow full-frame resimulation path.


## Run Order

1. Run the setup/config cell and confirm the uploaded checkpoint was found.
2. Run the embed/write cell once.
3. Optionally run the novelty tests.
4. Generate the adapter datasets.
5. Train `generic` and `fb_lara`.
6. Run NMSE first.
7. Run BER only if you want the full ablation curves.


In [ ]:
from pathlib import Path
import os
import sys
import zipfile
import subprocess

import torch

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/fb_lara_single')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT = WORK_ROOT / 'extracted_inputs'
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

AUTO_EXTRACT_ZIPS = True
RUN_TESTS = False
RUN_DATASET_BUILD = True
RUN_TRAIN_GENERIC = True
RUN_TRAIN_FB_LARA = True
RUN_EVAL_NMSE = True
RUN_EVAL_BER = False
DEVICE_OVERRIDE = 'cuda' if torch.cuda.is_available() else 'cpu'
ADAPTER_TRAIN_SIZE = 48000
ADAPTER_VAL_SIZE = 12000

def _search_direct(name: str):
    if not KAGGLE_INPUT_ROOT.exists():
        return None
    matches = sorted(KAGGLE_INPUT_ROOT.rglob(name))
    return matches[0] if matches else None

def _search_zip(name: str):
    if not AUTO_EXTRACT_ZIPS or not KAGGLE_INPUT_ROOT.exists():
        return None
    for zip_path in sorted(KAGGLE_INPUT_ROOT.rglob('*.zip')):
        with zipfile.ZipFile(zip_path, 'r') as zf:
            members = [member for member in zf.namelist() if Path(member).name == name]
            if not members:
                continue
            out_dir = EXTRACT_ROOT / zip_path.stem
            if not out_dir.exists() or not any(out_dir.rglob(name)):
                print(f'[setup] extracting {zip_path} to {out_dir}')
                zf.extractall(out_dir)
            matches = sorted(out_dir.rglob(name))
            if matches:
                return matches[0]
    return None

def find_artifact(name: str):
    direct = _search_direct(name)
    return direct if direct is not None else _search_zip(name)

BASELINE_CHECKPOINT = find_artifact('full_cnn_best.pt')
BASELINE_TRAIN_MANIFEST = find_artifact('train_full.json')
BASELINE_VAL_MANIFEST = find_artifact('val_full.json')

print({'device': DEVICE_OVERRIDE, 'checkpoint': str(BASELINE_CHECKPOINT) if BASELINE_CHECKPOINT else None})
print({'train_manifest': str(BASELINE_TRAIN_MANIFEST) if BASELINE_TRAIN_MANIFEST else None})
print({'val_manifest': str(BASELINE_VAL_MANIFEST) if BASELINE_VAL_MANIFEST else None})

assert BASELINE_CHECKPOINT is not None, 'Upload full_cnn_best.pt or a zip containing it.'


In [ ]:
from pathlib import Path
import sys

FILE_PAYLOADS = {
  "src/zakotfs_novelty/__init__.py": "\"\"\"Zak-OTFS novelty package.\"\"\"\n\nfrom .params import SystemConfig, load_config\n\n__all__ = [\"SystemConfig\", \"load_config\"]\n",
  "src/zakotfs_novelty/adapter.py": "from __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom .cnn_model import PaperCNN\nfrom .dataset import _split_seed, simulate_frame\nfrom .estimators import read_off_estimator\nfrom .operators import apply_support_operator\nfrom .params import SystemConfig, load_config\nfrom .utils import load_json, resolve_torch_device, results_dir, save_json\nfrom .waveform import spread_pilot\n\n\nFEATURE_CHANNEL_NAMES = (\n    \"raw_re\",\n    \"raw_im\",\n    \"base_re\",\n    \"base_im\",\n    \"alias_re\",\n    \"alias_im\",\n    \"raw_minus_base_re\",\n    \"raw_minus_base_im\",\n)\n\n\ndef load_frozen_backbone(config: SystemConfig, checkpoint_path: Path | None = None) -> tuple[PaperCNN, torch.device]:\n    if checkpoint_path is None:\n        checkpoint_cfg = config.raw.get(\"backbone\", {})\n        checkpoint_path = Path(str(checkpoint_cfg.get(\"checkpoint_path\", \"../logs/checkpoints/full_cnn_best.pt\")))\n    if not checkpoint_path.is_absolute():\n        checkpoint_path = (config.root / checkpoint_path).resolve()\n    device = resolve_torch_device(config)\n    model = PaperCNN().to(device=device, dtype=torch.float32)\n    payload = torch.load(checkpoint_path, map_location=device)\n    state_dict = payload[\"state_dict\"] if isinstance(payload, dict) and \"state_dict\" in payload else payload\n    model.load_state_dict(state_dict)\n    model.eval()\n    for param in model.parameters():\n        param.requires_grad_(False)\n    return model, device\n\n\ndef backbone_enhance_support(model: PaperCNN, support_input: np.ndarray, device: torch.device) -> np.ndarray:\n    with torch.no_grad():\n        x_re = torch.from_numpy(np.asarray(support_input.real[None, None, :, :], dtype=np.float32)).to(device)\n        x_im = torch.from_numpy(np.asarray(support_input.imag[None, None, :, :], dtype=np.float32)).to(device)\n        y_re = model(x_re).detach().cpu().numpy()[0, 0]\n        y_im = model(x_im).detach().cpu().numpy()[0, 0]\n    return (y_re + 1j * y_im).astype(np.complex64)\n\n\ndef synthesize_alias_prior(h_base: np.ndarray, spread_dd: np.ndarray, E_p: float, config: SystemConfig) -> np.ndarray:\n    y_syn = apply_support_operator(h_base, np.sqrt(E_p) * spread_dd, config)\n    h_syn = read_off_estimator(y_syn, spread_dd, E_p, config).support_input\n    return (h_syn - h_base).astype(np.complex64)\n\n\ndef build_fb_lara_features(h_raw: np.ndarray, h_base: np.ndarray, h_alias: np.ndarray) -> np.ndarray:\n    residual = h_raw - h_base\n    return np.stack(\n        [\n            h_raw.real,\n            h_raw.imag,\n            h_base.real,\n            h_base.imag,\n            h_alias.real,\n            h_alias.imag,\n            residual.real,\n            residual.imag,\n        ],\n        axis=0,\n    ).astype(np.float32)\n\n\ndef residual_target(h_true: np.ndarray, h_base: np.ndarray) -> np.ndarray:\n    delta = h_true - h_base\n    return np.stack([delta.real, delta.imag], axis=0).astype(np.float32)\n\n\ndef feature_channel_indices(adapter_kind: str) -> tuple[int, ...]:\n    kind = str(adapter_kind).lower()\n    if kind == \"fb_lara\":\n        return tuple(range(8))\n    if kind == \"generic\":\n        return (0, 1, 2, 3, 6, 7)\n    raise ValueError(f\"Unknown adapter kind '{adapter_kind}'\")\n\n\ndef adapter_dataset_sizes(config: SystemConfig, split: str) -> tuple[int, list[float], float]:\n    dataset_cfg = config.raw[\"adapter_dataset\"]\n    total = int(dataset_cfg[\"train_size_total\"] if split == \"train\" else dataset_cfg[\"val_size_total\"])\n    pdrs = list(map(float, dataset_cfg[\"training_pdr_db\"]))\n    snr = float(dataset_cfg[\"training_snr_db\"])\n    return total, pdrs, snr\n\n\ndef adapter_dataset_manifest(config: SystemConfig, split: str) -> dict:\n    total, pdrs, snr = adapter_dataset_sizes(config, split)\n    seed = _split_seed(config, split)\n    first = simulate_frame(config, \"bpsk\", snr, pdrs[0], np.random.default_rng(seed))\n    return {\n        \"generator\": \"memmap_fp16\",\n        \"split\": split,\n        \"seed\": seed,\n        \"size\": int(total),\n        \"snr_db\": snr,\n        \"pdr_db\": pdrs,\n        \"per_pdr\": int(total // len(pdrs)),\n        \"shape\": [int(first.support_input.shape[0]), int(first.support_input.shape[1])],\n        \"feature_channels\": list(FEATURE_CHANNEL_NAMES),\n        \"target_kind\": \"residual\",\n        \"backbone_checkpoint_path\": str(config.raw.get(\"backbone\", {}).get(\"checkpoint_path\", \"../logs/checkpoints/full_cnn_best.pt\")),\n    }\n\n\ndef _resolve_external_path(base: Path, value: str) -> Path:\n    target = Path(str(value))\n    if target.is_absolute():\n        return target\n    return (base / target).resolve()\n\n\ndef _baseline_manifest_path(config: SystemConfig, split: str) -> Path | None:\n    baseline_paths = config.raw.get(\"adapter_dataset\", {}).get(\"baseline_manifest_paths\", {}) or {}\n    candidate = baseline_paths.get(split)\n    if not candidate:\n        return None\n    path = _resolve_external_path(config.root, str(candidate))\n    return path if path.exists() else None\n\n\ndef _load_baseline_support_data(\n    manifest_path: Path,\n) -> tuple[dict, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:\n    meta = load_json(manifest_path)\n    if str(meta.get(\"generator\", \"\")) != \"memmap_fp16\":\n        raise ValueError(f\"Baseline manifest {manifest_path} is not a memmap_fp16 dataset\")\n    inputs_re = np.load(_resolve_external_path(manifest_path.parent, str(meta[\"inputs_re_path\"])), mmap_mode=\"r\")\n    inputs_im = np.load(_resolve_external_path(manifest_path.parent, str(meta[\"inputs_im_path\"])), mmap_mode=\"r\")\n    targets_re = np.load(_resolve_external_path(manifest_path.parent, str(meta[\"targets_re_path\"])), mmap_mode=\"r\")\n    targets_im = np.load(_resolve_external_path(manifest_path.parent, str(meta[\"targets_im_path\"])), mmap_mode=\"r\")\n    label_key = \"labels_path\" if \"labels_path\" in meta else \"pdr_labels_path\"\n    labels = np.load(_resolve_external_path(manifest_path.parent, str(meta[label_key])), mmap_mode=\"r\")\n    return meta, inputs_re, inputs_im, targets_re, targets_im, labels\n\n\nclass AdapterFeatureDataset(Dataset):\n    def __init__(self, manifest_path: Path, adapter_kind: str) -> None:\n        self.path = Path(manifest_path)\n        self.meta = load_json(self.path)\n        self.adapter_kind = str(adapter_kind).lower()\n        self.channel_indices = np.array(feature_channel_indices(self.adapter_kind), dtype=np.int64)\n        load_mode = \"r\"\n        self.inputs = np.load(self._resolve(\"inputs_path\"), mmap_mode=load_mode)\n        self.targets = np.load(self._resolve(\"targets_path\"), mmap_mode=load_mode)\n\n    def _resolve(self, key: str) -> str:\n        target = Path(str(self.meta[key]))\n        if target.is_absolute():\n            return str(target)\n        return str((self.path.parent / target).resolve())\n\n    def __len__(self) -> int:\n        return int(self.meta[\"size\"])\n\n    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:\n        x = np.asarray(self.inputs[index], dtype=np.float32)[self.channel_indices].copy()\n        y = np.asarray(self.targets[index], dtype=np.float32).copy()\n        return torch.from_numpy(x), torch.from_numpy(y)\n\n\ndef generate_adapter_dataset(config: SystemConfig, split: str, force: bool = False) -> Path:\n    out_dir = results_dir(config) / \"adapter_datasets\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    meta_path = out_dir / f\"{split}_adapter.json\"\n    if meta_path.exists() and not force:\n        meta = load_json(meta_path)\n        required = []\n        for key in [\"inputs_path\", \"targets_path\", \"pdr_labels_path\"]:\n            candidate = Path(str(meta[key]))\n            if not candidate.is_absolute():\n                candidate = (meta_path.parent / candidate).resolve()\n            required.append(candidate)\n        if all(path.exists() for path in required):\n            print(f\"[adapter-dataset] reusing {meta_path}\", flush=True)\n            return meta_path\n    manifest = adapter_dataset_manifest(config, split)\n    size = int(manifest[\"size\"])\n    H, W = map(int, manifest[\"shape\"])\n    per_pdr = int(manifest[\"per_pdr\"])\n    inputs_path = out_dir / f\"{split}_adapter_inputs.npy\"\n    targets_path = out_dir / f\"{split}_adapter_targets.npy\"\n    pdr_labels_path = out_dir / f\"{split}_adapter_pdr.npy\"\n    inputs = np.lib.format.open_memmap(inputs_path, mode=\"w+\", dtype=np.float16, shape=(size, 8, H, W))\n    targets = np.lib.format.open_memmap(targets_path, mode=\"w+\", dtype=np.float16, shape=(size, 2, H, W))\n    labels = np.lib.format.open_memmap(pdr_labels_path, mode=\"w+\", dtype=np.float32, shape=(size,))\n    backbone, device = load_frozen_backbone(config)\n    baseline_manifest = _baseline_manifest_path(config, split)\n    if baseline_manifest is not None:\n        base_meta, inputs_re, inputs_im, targets_re, targets_im, pdr_all = _load_baseline_support_data(baseline_manifest)\n        if int(base_meta[\"size\"]) < size:\n            raise ValueError(f\"Baseline manifest {baseline_manifest} has only {base_meta['size']} samples, expected at least {size}\")\n        spread_dd = spread_pilot(config)\n        print(\n            f\"[adapter-dataset] split={split} using baseline manifest {baseline_manifest} size={size} device={device}\",\n            flush=True,\n        )\n        for offset in range(size):\n            h_raw = np.asarray(inputs_re[offset], dtype=np.float32) + 1j * np.asarray(inputs_im[offset], dtype=np.float32)\n            h_true = np.asarray(targets_re[offset], dtype=np.float32) + 1j * np.asarray(targets_im[offset], dtype=np.float32)\n            pdr_db = float(pdr_all[offset])\n            E_p = float(10.0 ** (pdr_db / 10.0))\n            h_base = backbone_enhance_support(backbone, h_raw.astype(np.complex64), device)\n            h_alias = synthesize_alias_prior(h_base, spread_dd, E_p, config)\n            inputs[offset] = build_fb_lara_features(h_raw.astype(np.complex64), h_base, h_alias).astype(np.float16)\n            targets[offset] = residual_target(h_true.astype(np.complex64), h_base).astype(np.float16)\n            labels[offset] = np.float32(pdr_db)\n            if offset == 0 or (offset + 1) % 1000 == 0 or (offset + 1) == size:\n                print(f\"[adapter-dataset] split={split} baseline progress={offset + 1}/{size}\", flush=True)\n    else:\n        offset = 0\n        total, pdrs, snr = adapter_dataset_sizes(config, split)\n        if total != size:\n            raise RuntimeError(\"Adapter dataset size mismatch\")\n        for pdr_db in pdrs:\n            print(\n                f\"[adapter-dataset] split={split} pdr={pdr_db:.1f} samples={per_pdr} device={device}\",\n                flush=True,\n            )\n            for local_idx in range(per_pdr):\n                sample_seed = int(manifest[\"seed\"]) + int(offset)\n                frame = simulate_frame(config, \"bpsk\", snr, pdr_db, np.random.default_rng(sample_seed))\n                h_base = backbone_enhance_support(backbone, frame.support_input, device)\n                h_alias = synthesize_alias_prior(h_base, frame.spread_dd, frame.E_p, config)\n                inputs[offset] = build_fb_lara_features(frame.support_input, h_base, h_alias).astype(np.float16)\n                targets[offset] = residual_target(frame.support_true, h_base).astype(np.float16)\n                labels[offset] = np.float32(pdr_db)\n                offset += 1\n                if local_idx == 0 or (local_idx + 1) % 1000 == 0 or (local_idx + 1) == per_pdr:\n                    print(\n                        f\"[adapter-dataset] split={split} pdr={pdr_db:.1f} progress={local_idx + 1}/{per_pdr} total={offset}/{size}\",\n                        flush=True,\n                    )\n    inputs.flush()\n    targets.flush()\n    labels.flush()\n    materialized = dict(manifest)\n    materialized[\"inputs_path\"] = inputs_path.name\n    materialized[\"targets_path\"] = targets_path.name\n    materialized[\"pdr_labels_path\"] = pdr_labels_path.name\n    materialized[\"baseline_manifest_path\"] = str(baseline_manifest) if baseline_manifest is not None else None\n    save_json(meta_path, materialized)\n    print(f\"[adapter-dataset] saved manifest {meta_path}\", flush=True)\n    return meta_path\n\n\ndef load_adapter_train_config(path: Path | None = None) -> SystemConfig:\n    resolved = Path(\"novelty_paper/configs/adapter_train.yaml\") if path is None else Path(path)\n    return load_config(resolved)\n",
  "src/zakotfs_novelty/ambiguity.py": "from __future__ import annotations\n\nimport numpy as np\n\nfrom .params import SystemConfig\nfrom .utils import centered_coord_to_index\n\n\ndef quasi_periodic_get(arr: np.ndarray, k: int, l: int) -> np.complex64:\n    M, N = arr.shape\n    base_k = k % M\n    base_l = l % N\n    wrap_k = (k - base_k) // M\n    phase = np.exp(1j * 2 * np.pi * wrap_k * base_l / N)\n    return np.complex64(arr[base_k, base_l] * phase)\n\n\ndef cross_ambiguity(a: np.ndarray, b: np.ndarray) -> np.ndarray:\n    M, N = a.shape\n    Q = M * N\n    out = np.zeros((M, N), dtype=np.complex64)\n    kp = np.arange(M, dtype=int)[:, None]\n    lp = np.arange(N, dtype=int)[None, :]\n    for k in range(M):\n        shifted_k = kp - k\n        base_k = np.mod(shifted_k, M)\n        wrap_k = np.floor_divide(shifted_k - base_k, M)\n        for l in range(N):\n            shifted_l = lp - l\n            base_l = np.mod(shifted_l, N)\n            phase_qp = np.exp(1j * 2 * np.pi * wrap_k * base_l / N)\n            b_shifted = b[base_k, base_l] * phase_qp\n            phase = np.exp(-1j * 2 * np.pi * l * shifted_k / Q)\n            out[k, l] = np.sum(a * np.conjugate(b_shifted) * phase)\n    return out\n\n\ndef cross_ambiguity_window(a: np.ndarray, b: np.ndarray, k_values: list[int] | range | np.ndarray, l_values: list[int] | range | np.ndarray) -> np.ndarray:\n    M, N = a.shape\n    Q = M * N\n    ks = list(k_values)\n    ls = list(l_values)\n    out = np.zeros((len(ks), len(ls)), dtype=np.complex64)\n    kp = np.arange(M, dtype=int)[:, None]\n    lp = np.arange(N, dtype=int)[None, :]\n    for i, k in enumerate(ks):\n        shifted_k = kp - k\n        base_k = np.mod(shifted_k, M)\n        wrap_k = np.floor_divide(shifted_k - base_k, M)\n        for j, l in enumerate(ls):\n            shifted_l = lp - l\n            base_l = np.mod(shifted_l, N)\n            phase_qp = np.exp(1j * 2 * np.pi * wrap_k * base_l / N)\n            b_shifted = b[base_k, base_l] * phase_qp\n            phase = np.exp(-1j * 2 * np.pi * l * shifted_k / Q)\n            out[i, j] = np.sum(a * np.conjugate(b_shifted) * phase)\n    return out\n\n\ndef centered_cross_ambiguity(a: np.ndarray, b: np.ndarray, config: SystemConfig) -> np.ndarray:\n    raw = cross_ambiguity(a, b)\n    out = np.zeros_like(raw)\n    for k in range(config.M):\n        ck = k if k <= config.M // 2 else k - config.M\n        for l in range(config.N):\n            cl = l if l <= config.N // 2 else l - config.N\n            out[centered_coord_to_index(ck, config.M), centered_coord_to_index(cl, config.N)] = raw[k, l]\n    return out\n\n\ndef self_ambiguity_support(config: SystemConfig, spread_pilot: np.ndarray, tol: float = 1e-5) -> np.ndarray:\n    amb = cross_ambiguity(spread_pilot, spread_pilot)\n    return (np.abs(amb) > tol).astype(np.uint8)\n",
  "src/zakotfs_novelty/channel.py": "from __future__ import annotations\n\nimport numpy as np\n\nfrom .compat import dataclass_slots, strict_zip\nfrom .lattice import derive_support_geometry, embed_support_image\nfrom .params import SystemConfig\nfrom .pulses import (\n    effective_pulse_kernel,\n    gs_delay_autocorrelation,\n    gs_delay_overlap_exact,\n    gs_doppler_autocorrelation,\n    gs_doppler_overlap_exact,\n)\nfrom .utils import complex_noise\n\n\n@dataclass_slots()\nclass PhysicalChannel:\n    gains: np.ndarray\n    delays_s: np.ndarray\n    dopplers_hz: np.ndarray\n    thetas_rad: np.ndarray\n\n\ndef sample_vehicular_a_channel(config: SystemConfig, rng: np.random.Generator) -> PhysicalChannel:\n    c = config.channel\n    powers_lin = 10.0 ** (np.array(c[\"relative_powers_db\"], dtype=float) / 10.0)\n    powers_lin = powers_lin / np.sum(powers_lin)\n    gains = np.sqrt(powers_lin / 2.0) * (\n        rng.standard_normal(len(powers_lin)) + 1j * rng.standard_normal(len(powers_lin))\n    )\n    delays_s = np.array(c[\"delays_s\"], dtype=float)\n    thetas = rng.uniform(0.0, 2.0 * np.pi, size=len(powers_lin))\n    dopplers_hz = c[\"max_doppler_hz\"] * np.cos(thetas)\n    return PhysicalChannel(gains=gains.astype(np.complex64), delays_s=delays_s, dopplers_hz=dopplers_hz.astype(float), thetas_rad=thetas)\n\n\ndef _support_axes(config: SystemConfig) -> tuple[np.ndarray, np.ndarray]:\n    g = derive_support_geometry(config)\n    delay_s = np.arange(g.k_min, g.k_max + 1, dtype=float) / config.B\n    doppler_hz = np.arange(g.l_min, g.l_max + 1, dtype=float) / config.T\n    return delay_s, doppler_hz\n\n\ndef effective_channel_method(config: SystemConfig) -> str:\n    return str(config.raw.get(\"numerics\", {}).get(\"effective_channel_method\", \"fast\")).lower()\n\n\ndef effective_channel_support_reference(channel: PhysicalChannel, config: SystemConfig) -> np.ndarray:\n    # Reference implementation of Eq. (26) in the GS-filter paper. It is slower but\n    # captures the path-dependent phase and exact matched-filter overlaps.\n    delay_s, doppler_hz = _support_axes(config)\n    heff = np.zeros((delay_s.size, doppler_hz.size), dtype=np.complex64)\n    for gain, tau_i, nu_i in strict_zip(channel.gains, channel.delays_s, channel.dopplers_hz):\n        i1 = gs_delay_overlap_exact(delay_s, tau_i, nu_i, config).astype(np.complex64)\n        i2 = gs_doppler_overlap_exact(delay_s, doppler_hz, nu_i, config).astype(np.complex64)\n        path_phase = np.exp(1j * 2 * np.pi * nu_i * (delay_s - tau_i)).astype(np.complex64)\n        heff += gain * path_phase[:, None] * i1[:, None] * i2\n    return heff\n\n\ndef effective_channel_support_fast(channel: PhysicalChannel, config: SystemConfig) -> np.ndarray:\n    # Fast crystalline-regime approximation consistent with Corollary 3 in the Zak-OTFS\n    # receiver paper: pathwise DD localization with phase exp(j 2 pi (nu tau - nu_i tau_i)).\n    delay_s, doppler_hz = _support_axes(config)\n    heff = np.zeros((delay_s.size, doppler_hz.size), dtype=np.complex64)\n    for gain, tau_i, nu_i in strict_zip(channel.gains, channel.delays_s, channel.dopplers_hz):\n        delay_auto = gs_delay_autocorrelation(delay_s - tau_i, config).astype(np.complex64)\n        doppler_auto = gs_doppler_autocorrelation(doppler_hz - nu_i, config).astype(np.complex64)\n        path_phase = np.exp(1j * 2 * np.pi * (delay_s[:, None] * doppler_hz[None, :] - nu_i * tau_i)).astype(np.complex64)\n        heff += gain * path_phase * delay_auto[:, None] * doppler_auto[None, :]\n    return heff\n\n\ndef effective_channel_support_envelope(channel: PhysicalChannel, config: SystemConfig) -> np.ndarray:\n    delay_s, doppler_hz = _support_axes(config)\n    heff = np.zeros((delay_s.size, doppler_hz.size), dtype=np.complex64)\n    for gain, tau_i, nu_i in strict_zip(channel.gains, channel.delays_s, channel.dopplers_hz):\n        envelope = effective_pulse_kernel(delay_s[:, None] - tau_i, doppler_hz[None, :] - nu_i, config)\n        phase = np.exp(1j * 2 * np.pi * (delay_s[:, None] * doppler_hz[None, :] - nu_i * tau_i))\n        heff += gain * envelope * phase.astype(np.complex64)\n    return heff\n\n\ndef effective_channel_support(channel: PhysicalChannel, config: SystemConfig) -> np.ndarray:\n    method = effective_channel_method(config)\n    if method == \"reference\":\n        return effective_channel_support_reference(channel, config)\n    if method == \"envelope\":\n        return effective_channel_support_envelope(channel, config)\n    return effective_channel_support_fast(channel, config)\n\n\ndef effective_channel_taps_reference(channel: PhysicalChannel, config: SystemConfig) -> np.ndarray:\n    return embed_support_image(effective_channel_support_reference(channel, config), config)\n\n\ndef effective_channel_taps_fast(channel: PhysicalChannel, config: SystemConfig) -> np.ndarray:\n    return embed_support_image(effective_channel_support_fast(channel, config), config)\n\n\ndef effective_channel_taps(channel: PhysicalChannel, config: SystemConfig) -> np.ndarray:\n    return embed_support_image(effective_channel_support(channel, config), config)\n\n\ndef add_awgn(signal: np.ndarray, noise_variance: float, rng: np.random.Generator) -> np.ndarray:\n    return signal + complex_noise(signal.shape, noise_variance, rng)\n",
  "src/zakotfs_novelty/cnn_model.py": "from __future__ import annotations\n\nimport torch\nfrom torch import nn\n\n\nclass PaperCNN(nn.Module):\n    def __init__(self) -> None:\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(1, 64, kernel_size=(27, 27), stride=1, padding=\"same\"),\n            nn.ReLU(),\n            nn.Conv2d(64, 32, kernel_size=(9, 9), stride=1, padding=\"same\"),\n            nn.ReLU(),\n            nn.Conv2d(32, 32, kernel_size=(5, 5), stride=1, padding=\"same\"),\n            nn.ReLU(),\n            nn.Conv2d(32, 1, kernel_size=(15, 15), stride=1, padding=\"same\"),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n    @property\n    def num_parameters(self) -> int:\n        return sum(param.numel() for param in self.parameters())\n\n\nclass ResidualAdapter(nn.Module):\n    def __init__(self, input_channels: int) -> None:\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(input_channels, 16, kernel_size=1, stride=1),\n            nn.ReLU(),\n            nn.Conv2d(16, 16, kernel_size=5, stride=1, padding=2),\n            nn.ReLU(),\n            nn.Conv2d(16, 2, kernel_size=1, stride=1),\n        )\n        self._zero_init_last_layer()\n\n    def _zero_init_last_layer(self) -> None:\n        last = self.net[-1]\n        if isinstance(last, nn.Conv2d):\n            nn.init.zeros_(last.weight)\n            if last.bias is not None:\n                nn.init.zeros_(last.bias)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n    @property\n    def num_parameters(self) -> int:\n        return sum(param.numel() for param in self.parameters())\n\n\nclass GenericResidualAdapter(ResidualAdapter):\n    def __init__(self) -> None:\n        super().__init__(input_channels=6)\n\n\nclass LatticeAliasAdapter(ResidualAdapter):\n    def __init__(self) -> None:\n        super().__init__(input_channels=8)\n",
  "src/zakotfs_novelty/compat.py": "from __future__ import annotations\n\nimport sys\nfrom dataclasses import dataclass\n\n\ndef dataclass_slots(*args, **kwargs):\n    if sys.version_info >= (3, 10):\n        kwargs.setdefault(\"slots\", True)\n    else:\n        kwargs.pop(\"slots\", None)\n    return dataclass(*args, **kwargs)\n\n\ndef strict_zip(*iterables):\n    if sys.version_info >= (3, 10):\n        return zip(*iterables, strict=True)\n    return zip(*iterables)\n",
  "src/zakotfs_novelty/dataset.py": "from __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom .compat import dataclass_slots\nfrom .channel import add_awgn, effective_channel_method, effective_channel_support, sample_vehicular_a_channel\nfrom .estimators import read_off_estimator, threshold_readoff\nfrom .lattice import crop_support, embed_support_image\nfrom .modulation import hard_demodulate\nfrom .params import SystemConfig\nfrom .utils import results_dir, save_json, set_global_seed, snr_to_noise_variance\nfrom .waveform import data_symbols, data_waveform, spread_pilot, superimposed_frame\nfrom .operators import apply_heff_operator, apply_support_operator\n\n\n@dataclass_slots()\nclass FrameBundle:\n    modulation: str\n    symbols: np.ndarray\n    bits: np.ndarray\n    data_dd: np.ndarray\n    spread_dd: np.ndarray\n    x_dd: np.ndarray\n    physical_channel: object\n    h_eff_support: np.ndarray\n    h_eff: np.ndarray\n    y_clean: np.ndarray\n    y_dd: np.ndarray\n    ambiguity: np.ndarray\n    h_hat_support_raw: np.ndarray\n    h_hat_support_thr: np.ndarray\n    h_hat_raw: np.ndarray\n    h_hat_thr: np.ndarray\n    support_input: np.ndarray\n    support_true: np.ndarray\n    E_d: float\n    E_p: float\n    rho_d: float\n    rho_p: float\n    noise_variance: float\n\n\ndef _split_seed(config: SystemConfig, split: str) -> int:\n    offsets = {\"train\": 0, \"val\": 10_000, \"test\": 20_000, \"eval\": 30_000, \"smoke\": 40_000}\n    return config.seed + offsets.get(split, 50_000)\n\n\ndef simulate_frame(\n    config: SystemConfig,\n    modulation: str,\n    data_snr_db: float,\n    pdr_db: float,\n    rng: np.random.Generator,\n) -> FrameBundle:\n    symbols, bits = data_symbols(modulation, config, rng)\n    data_dd = data_waveform(symbols)\n    spread_dd = spread_pilot(config)\n    E_d = 1.0\n    E_p = 10.0 ** (pdr_db / 10.0) * E_d\n    noise_variance = snr_to_noise_variance(E_d, data_snr_db, config.Q)\n    rho_d = E_d / (config.Q * noise_variance)\n    rho_p = E_p / (config.Q * noise_variance)\n    x_dd = superimposed_frame(data_dd, spread_dd, E_d=E_d, E_p=E_p)\n    physical_channel = sample_vehicular_a_channel(config, rng)\n    h_eff_support = effective_channel_support(physical_channel, config)\n    h_eff = embed_support_image(h_eff_support, config)\n    y_clean = apply_support_operator(h_eff_support, x_dd, config)\n    y_dd = add_awgn(y_clean, noise_variance, rng)\n    est = read_off_estimator(y_dd, spread_dd, E_p, config)\n    h_hat_support_raw = est.support_input\n    h_hat_support_thr = threshold_readoff(h_hat_support_raw, rho_d, rho_p, config)\n    h_hat_raw = embed_support_image(h_hat_support_raw, config)\n    h_hat_thr = embed_support_image(h_hat_support_thr, config)\n    support_input = h_hat_support_raw\n    support_true = h_eff_support\n    return FrameBundle(\n        modulation=modulation,\n        symbols=symbols,\n        bits=bits,\n        data_dd=data_dd,\n        spread_dd=spread_dd,\n        x_dd=x_dd,\n        physical_channel=physical_channel,\n        h_eff_support=h_eff_support,\n        h_eff=h_eff,\n        y_clean=y_clean,\n        y_dd=y_dd,\n        ambiguity=est.ambiguity,\n        h_hat_support_raw=h_hat_support_raw,\n        h_hat_support_thr=h_hat_support_thr,\n        h_hat_raw=h_hat_raw,\n        h_hat_thr=h_hat_thr,\n        support_input=support_input,\n        support_true=support_true,\n        E_d=E_d,\n        E_p=E_p,\n        rho_d=rho_d,\n        rho_p=rho_p,\n        noise_variance=noise_variance,\n    )\n\n\ndef dataset_sizes(config: SystemConfig, split: str, mode: str) -> tuple[int, list[float], float]:\n    if mode == \"smoke\":\n        dataset_cfg = config.raw.get(\"smoke\", {}).get(\"dataset\", {})\n    else:\n        dataset_cfg = config.raw[\"dataset\"]\n    total = int(dataset_cfg[\"train_size_total\"] if split == \"train\" else dataset_cfg[\"val_size_total\"])\n    pdrs = list(map(float, config.raw[\"dataset\"][\"training_pdr_db\"]))\n    snr = float(config.raw[\"dataset\"][\"training_snr_db\"])\n    return total, pdrs, snr\n\n\ndef dataset_manifest(config: SystemConfig, split: str, mode: str) -> dict:\n    total, pdrs, snr = dataset_sizes(config, split, mode)\n    seed = _split_seed(config, split) + (0 if mode == \"full\" else 100_000)\n    per_pdr = total // len(pdrs)\n    first_rng = np.random.default_rng(seed)\n    first = simulate_frame(config, \"bpsk\", snr, pdrs[0], first_rng)\n    return {\n        \"generator\": \"on_the_fly\" if mode == \"full\" else \"npz\",\n        \"split\": split,\n        \"mode\": mode,\n        \"seed\": seed,\n        \"size\": int(per_pdr * len(pdrs)),\n        \"snr_db\": snr,\n        \"pdr_db\": pdrs,\n        \"shape\": [int(first.support_input.shape[0]), int(first.support_input.shape[1])],\n        \"per_pdr\": int(per_pdr),\n        \"modulation\": \"bpsk\",\n        \"effective_channel_method\": effective_channel_method(config),\n    }\n\n\nclass GeneratedSupportDataset(Dataset):\n    def __init__(self, config: SystemConfig, manifest: dict) -> None:\n        self.config = config\n        self.meta = manifest\n\n    def __len__(self) -> int:\n        return int(self.meta[\"size\"])\n\n    def __getitem__(self, index: int) -> tuple[np.ndarray, np.ndarray, np.float32]:\n        pdrs = list(map(float, self.meta[\"pdr_db\"]))\n        per_pdr = int(self.meta[\"per_pdr\"])\n        pdr_idx = min(index // per_pdr, len(pdrs) - 1)\n        pdr_db = pdrs[pdr_idx]\n        sample_seed = int(self.meta[\"seed\"]) + int(index)\n        frame = simulate_frame(\n            self.config,\n            str(self.meta.get(\"modulation\", \"bpsk\")),\n            float(self.meta[\"snr_db\"]),\n            pdr_db,\n            np.random.default_rng(sample_seed),\n        )\n        return frame.support_input.astype(np.complex64), frame.support_true.astype(np.complex64), np.float32(pdr_db)\n\n\ndef _materialize_full_dataset(config: SystemConfig, split: str, manifest: dict, out_dir: Path) -> Path:\n    meta_path = out_dir / f\"{split}_full.json\"\n    inputs_re_path = out_dir / f\"{split}_full_inputs_re.npy\"\n    inputs_im_path = out_dir / f\"{split}_full_inputs_im.npy\"\n    targets_re_path = out_dir / f\"{split}_full_targets_re.npy\"\n    targets_im_path = out_dir / f\"{split}_full_targets_im.npy\"\n    labels_path = out_dir / f\"{split}_full_pdr.npy\"\n    size = int(manifest[\"size\"])\n    H, W = map(int, manifest[\"shape\"])\n    batch_size = int(config.raw[\"dataset\"].get(\"materialize_batch_size\", 128))\n    num_workers = int(config.raw[\"dataset\"].get(\"materialize_num_workers\", config.raw[\"dataset\"].get(\"num_workers\", 0)))\n    print(\n        f\"[dataset] materializing split={split} samples={size} shape=({H},{W}) \"\n        f\"batch_size={batch_size} num_workers={num_workers}\",\n        flush=True,\n    )\n    ds = GeneratedSupportDataset(config, manifest)\n    loader = DataLoader(\n        ds,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=num_workers,\n        persistent_workers=num_workers > 0,\n    )\n    inputs_re = np.lib.format.open_memmap(inputs_re_path, mode=\"w+\", dtype=np.float16, shape=(size, H, W))\n    inputs_im = np.lib.format.open_memmap(inputs_im_path, mode=\"w+\", dtype=np.float16, shape=(size, H, W))\n    targets_re = np.lib.format.open_memmap(targets_re_path, mode=\"w+\", dtype=np.float16, shape=(size, H, W))\n    targets_im = np.lib.format.open_memmap(targets_im_path, mode=\"w+\", dtype=np.float16, shape=(size, H, W))\n    labels = np.lib.format.open_memmap(labels_path, mode=\"w+\", dtype=np.float32, shape=(size,))\n    offset = 0\n    total_batches = len(loader)\n    for batch_idx, (x, y, pdr) in enumerate(loader, start=1):\n        batch = int(x.shape[0])\n        x_np = x.detach().cpu().numpy()\n        y_np = y.detach().cpu().numpy()\n        labels[offset : offset + batch] = pdr.detach().cpu().numpy().astype(np.float32)\n        inputs_re[offset : offset + batch] = x_np.real.astype(np.float16)\n        inputs_im[offset : offset + batch] = x_np.imag.astype(np.float16)\n        targets_re[offset : offset + batch] = y_np.real.astype(np.float16)\n        targets_im[offset : offset + batch] = y_np.imag.astype(np.float16)\n        offset += batch\n        if batch_idx == 1 or batch_idx % 20 == 0 or batch_idx == total_batches:\n            print(\n                f\"[dataset] split={split} materialize batch {batch_idx}/{total_batches} samples_written={offset}/{size}\",\n                flush=True,\n            )\n    inputs_re.flush()\n    inputs_im.flush()\n    targets_re.flush()\n    targets_im.flush()\n    labels.flush()\n    materialized = dict(manifest)\n    materialized[\"generator\"] = \"memmap_fp16\"\n    materialized[\"inputs_re_path\"] = inputs_re_path.name\n    materialized[\"inputs_im_path\"] = inputs_im_path.name\n    materialized[\"targets_re_path\"] = targets_re_path.name\n    materialized[\"targets_im_path\"] = targets_im_path.name\n    materialized[\"labels_path\"] = labels_path.name\n    save_json(meta_path, materialized)\n    print(f\"[dataset] saved materialized manifest {meta_path}\", flush=True)\n    return meta_path\n\n\ndef generate_dataset(config: SystemConfig, split: str, mode: str = \"full\", force: bool = False) -> Path:\n    if \"dataset\" not in config.raw:\n        raise ValueError(\"Dataset config missing\")\n    out_dir = results_dir(config) / \"datasets\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    out_path = out_dir / f\"{split}_{mode}.npz\"\n    meta_path = out_dir / f\"{split}_{mode}.json\"\n    if mode == \"full\":\n        if meta_path.exists() and not force:\n            meta = None\n            try:\n                from .utils import load_json\n\n                meta = load_json(meta_path)\n            except Exception:\n                meta = None\n            if meta and meta.get(\"generator\") == \"memmap_fp16\":\n                required = []\n                for key in [\"inputs_re_path\", \"inputs_im_path\", \"targets_re_path\", \"targets_im_path\", \"labels_path\"]:\n                    candidate = Path(str(meta[key]))\n                    if not candidate.is_absolute():\n                        candidate = (meta_path.parent / candidate).resolve()\n                    required.append(candidate)\n                if all(path.exists() for path in required):\n                    print(f\"[dataset] reusing materialized full dataset {meta_path}\", flush=True)\n                    return meta_path\n            elif meta and meta.get(\"generator\") == \"on_the_fly\" and not bool(config.raw[\"dataset\"].get(\"save_npz\", True)):\n                print(f\"[dataset] reusing full manifest {meta_path}\", flush=True)\n                return meta_path\n        manifest = dataset_manifest(config, split, mode)\n        if bool(config.raw[\"dataset\"].get(\"save_npz\", True)):\n            return _materialize_full_dataset(config, split, manifest, out_dir)\n        print(f\"[dataset] writing full manifest for split={split} to {meta_path}\", flush=True)\n        save_json(meta_path, manifest)\n        return meta_path\n    if out_path.exists() and meta_path.exists() and not force:\n        print(f\"[dataset] reusing materialized dataset {out_path}\", flush=True)\n        return out_path\n    manifest = dataset_manifest(config, split, mode)\n    seed = int(manifest[\"seed\"])\n    pdrs = list(map(float, manifest[\"pdr_db\"]))\n    snr = float(manifest[\"snr_db\"])\n    per_pdr = int(manifest[\"per_pdr\"])\n    H, W = map(int, manifest[\"shape\"])\n    inputs = np.zeros((per_pdr * len(pdrs), H, W), dtype=np.complex64)\n    targets = np.zeros_like(inputs)\n    pdr_labels = np.zeros(inputs.shape[0], dtype=np.float32)\n    index = 0\n    rng = np.random.default_rng(seed)\n    for pdr_db in pdrs:\n        print(f\"[dataset] split={split} mode={mode} generating pdr={pdr_db} dB samples={per_pdr}\", flush=True)\n        for _ in range(per_pdr):\n            frame = simulate_frame(config, \"bpsk\", snr, pdr_db, rng)\n            inputs[index] = frame.support_input\n            targets[index] = frame.support_true\n            pdr_labels[index] = pdr_db\n            index += 1\n    np.savez_compressed(out_path, inputs=inputs, targets=targets, pdr_db=pdr_labels)\n    manifest[\"generator\"] = \"npz\"\n    save_json(meta_path, manifest)\n    print(f\"[dataset] saved dataset {out_path}\", flush=True)\n    print(f\"[dataset] saved metadata {meta_path}\", flush=True)\n    return out_path\n\n\ndef detect_bits_from_data_symbols(symbols_hat: np.ndarray, modulation: str) -> np.ndarray:\n    _, bits = hard_demodulate(symbols_hat, modulation)\n    return bits\n",
  "src/zakotfs_novelty/diagnostics.py": "from __future__ import annotations\n\nfrom dataclasses import asdict\n\nimport numpy as np\n\nfrom .ambiguity import cross_ambiguity\nfrom .compat import dataclass_slots\nfrom .dataset import simulate_frame\nfrom .lattice import crop_support\nfrom .operators import apply_heff_operator\nfrom .params import SystemConfig\n\n\n@dataclass_slots()\nclass ErrorDecomposition:\n    effective_energy: float\n    alias_energy: float\n    data_interference_energy: float\n    noise_energy: float\n    total_energy: float\n\n\ndef error_decomposition(config: SystemConfig, seed_offset: int = 0) -> tuple[dict[str, np.ndarray], ErrorDecomposition]:\n    rng = np.random.default_rng(config.seed + 500 + seed_offset)\n    frame = simulate_frame(config, \"bpsk\", 15.0, 5.0, rng)\n    A_xsxs = cross_ambiguity(frame.spread_dd, frame.spread_dd)\n    A_xdxs = cross_ambiguity(frame.data_dd, frame.spread_dd)\n    noise_dd = frame.y_dd - frame.y_clean\n    A_nxs = cross_ambiguity(noise_dd, frame.spread_dd)\n    effective_term = np.sqrt(frame.E_p) * frame.h_eff\n    alias_full = np.sqrt(frame.E_p) * (apply_heff_operator(frame.h_eff, A_xsxs, config) - frame.h_eff)\n    data_term = np.sqrt(frame.E_d) * apply_heff_operator(frame.h_eff, A_xdxs, config)\n    noise_term = A_nxs\n    components = {\n        \"effective_term\": crop_support(effective_term, config),\n        \"alias_term\": crop_support(alias_full, config),\n        \"data_term\": crop_support(data_term, config),\n        \"noise_term\": crop_support(noise_term, config),\n        \"read_off\": frame.support_input,\n        \"truth\": frame.support_true,\n    }\n    energies = {name: float(np.sum(np.abs(value) ** 2)) for name, value in components.items() if name.endswith(\"_term\")}\n    summary = ErrorDecomposition(\n        effective_energy=energies[\"effective_term\"],\n        alias_energy=energies[\"alias_term\"],\n        data_interference_energy=energies[\"data_term\"],\n        noise_energy=energies[\"noise_term\"],\n        total_energy=float(sum(energies.values())),\n    )\n    return components, summary\n\n\ndef decomposition_dict(summary: ErrorDecomposition) -> dict[str, float]:\n    return asdict(summary)\n",
  "src/zakotfs_novelty/estimators.py": "from __future__ import annotations\n\nimport numpy as np\n\nfrom .ambiguity import cross_ambiguity, cross_ambiguity_window\nfrom .compat import dataclass_slots\nfrom .lattice import crop_support, derive_support_geometry, embed_support_image\nfrom .operators import apply_heff_operator, apply_support_operator\nfrom .params import SystemConfig\n\n\n@dataclass_slots()\nclass EstimationBundle:\n    ambiguity: np.ndarray\n    h_hat_raw: np.ndarray\n    h_hat_thresholded: np.ndarray\n    support_input: np.ndarray\n    support_true: np.ndarray\n\n\ndef read_off_estimator(y_dd: np.ndarray, xs_dd: np.ndarray, E_p: float, config: SystemConfig) -> EstimationBundle:\n    ambiguity = cross_ambiguity(y_dd, xs_dd) / np.sqrt(E_p)\n    g = derive_support_geometry(config)\n    support_input = cross_ambiguity_window(y_dd, xs_dd, range(g.k_min, g.k_max + 1), range(g.l_min, g.l_max + 1)) / np.sqrt(E_p)\n    h_hat_raw = embed_support_image(support_input, config)\n    return EstimationBundle(\n        ambiguity=ambiguity,\n        h_hat_raw=h_hat_raw,\n        h_hat_thresholded=h_hat_raw.copy(),\n        support_input=support_input,\n        support_true=np.zeros_like(support_input),\n    )\n\n\ndef threshold_readoff(h_hat_raw: np.ndarray, rho_d: float, rho_p: float, config: SystemConfig) -> np.ndarray:\n    sigma = np.sqrt((1.0 / config.Q) * (1.0 + rho_d / rho_p))\n    threshold = config.estimation[\"threshold_multiplier\"] * sigma\n    out = h_hat_raw.copy()\n    out[np.abs(out) < threshold] = 0.0\n    return out\n\n\ndef pilot_cancellation(y_dd: np.ndarray, h_est: np.ndarray, xs_dd: np.ndarray) -> np.ndarray:\n    raise RuntimeError(\"pilot_cancellation now requires pilot_cancellation_with_config\")\n\n\ndef pilot_cancellation_with_config(y_dd: np.ndarray, h_est: np.ndarray, xs_dd: np.ndarray, config: SystemConfig) -> np.ndarray:\n    op = apply_heff_operator if h_est.shape == (config.M, config.N) else apply_support_operator\n    return y_dd - op(h_est, xs_dd, config)\n\n\ndef support_images(h_eff: np.ndarray, h_hat: np.ndarray, config: SystemConfig) -> tuple[np.ndarray, np.ndarray]:\n    return crop_support(h_hat, config), crop_support(h_eff, config)\n\n\ndef embed_cnn_output(support_img: np.ndarray, config: SystemConfig) -> np.ndarray:\n    return embed_support_image(support_img, config)\n",
  "src/zakotfs_novelty/evaluation.py": "from __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom .adapter import build_fb_lara_features, feature_channel_indices, synthesize_alias_prior\nfrom .dataset import simulate_frame\nfrom .estimators import pilot_cancellation_with_config\nfrom .metrics import ber_from_bits, nmse\nfrom .mmse import mmse_dense, mmse_dense_torch, mmse_iterative\nfrom .modulation import hard_demodulate\nfrom .params import SystemConfig\nfrom .plotting import curve_plot_columns, save_curve_plot\nfrom .utils import bootstrap_mean_ci, resolve_torch_device, results_dir, save_json, wilson_ci\n\n\ndef cnn_enhance_support(model: torch.nn.Module, support_input: np.ndarray, device: torch.device) -> np.ndarray:\n    with torch.no_grad():\n        x_re = torch.from_numpy(np.asarray(support_input.real[None, None, :, :], dtype=np.float32)).to(device)\n        x_im = torch.from_numpy(np.asarray(support_input.imag[None, None, :, :], dtype=np.float32)).to(device)\n        y_re = model(x_re).detach().cpu().numpy()[0, 0]\n        y_im = model(x_im).detach().cpu().numpy()[0, 0]\n    return (y_re + 1j * y_im).astype(np.complex64)\n\n\ndef _adapter_enhance_support(\n    adapter_model: torch.nn.Module,\n    adapter_kind: str,\n    support_input: np.ndarray,\n    h_base: np.ndarray,\n    h_alias: np.ndarray,\n) -> np.ndarray:\n    device = next(adapter_model.parameters()).device\n    features = build_fb_lara_features(support_input, h_base, h_alias)\n    channel_idx = np.array(feature_channel_indices(adapter_kind), dtype=np.int64)\n    with torch.no_grad():\n        x = torch.from_numpy(np.asarray(features[channel_idx][None], dtype=np.float32)).to(device)\n        delta = adapter_model(x).detach().cpu().numpy()[0]\n    return (h_base + delta[0] + 1j * delta[1]).astype(np.complex64)\n\n\ndef estimate_channels(\n    frame,\n    config: SystemConfig,\n    backbone: torch.nn.Module | None = None,\n    adapters: dict[str, torch.nn.Module] | None = None,\n) -> dict[str, np.ndarray]:\n    estimates = {\n        \"conventional_raw\": frame.h_hat_support_raw,\n        \"conventional\": frame.h_hat_support_thr,\n        \"perfect\": frame.h_eff_support,\n    }\n    if backbone is not None:\n        device = next(backbone.parameters()).device\n        h_base = cnn_enhance_support(backbone, frame.support_input, device)\n        estimates[\"cnn\"] = h_base\n        if adapters:\n            h_alias = synthesize_alias_prior(h_base, frame.spread_dd, frame.E_p, config)\n            for adapter_kind, adapter_model in adapters.items():\n                estimates[str(adapter_kind)] = _adapter_enhance_support(\n                    adapter_model,\n                    str(adapter_kind),\n                    frame.support_input,\n                    h_base,\n                    h_alias,\n                )\n    return estimates\n\n\ndef _default_solver_name(config: SystemConfig, mode: str, point_cfg: dict) -> str:\n    solver = str(point_cfg.get(\"solver\", \"\")).lower()\n    if solver:\n        return solver\n    use_dense = bool(point_cfg.get(\"use_dense\", mode == \"smoke\"))\n    if use_dense:\n        if resolve_torch_device(config).type != \"cpu\":\n            return \"dense_torch\"\n        return \"dense\"\n    return \"iterative\"\n\n\ndef detect_frame(frame, h_est: np.ndarray, config: SystemConfig, solver: str = \"iterative\") -> tuple[np.ndarray, np.ndarray]:\n    y_data = pilot_cancellation_with_config(frame.y_dd, h_est, np.sqrt(frame.E_p) * frame.spread_dd, config)\n    solver_name = str(solver).lower()\n    if solver_name == \"dense_torch\":\n        device = resolve_torch_device(config)\n        x_hat_dd = mmse_dense_torch(y_data, h_est, frame.noise_variance, config, E_d=frame.E_d, device=device)\n    elif solver_name == \"dense\":\n        x_hat_dd = mmse_dense(y_data, h_est, frame.noise_variance, config, E_d=frame.E_d)\n    elif solver_name == \"iterative\":\n        x_hat_dd = mmse_iterative(y_data, h_est, frame.noise_variance, config, E_d=frame.E_d)\n    else:\n        raise ValueError(f\"Unknown MMSE solver '{solver}'\")\n    symbols_hat = x_hat_dd * np.sqrt(config.Q) / np.sqrt(frame.E_d)\n    decided, bits = hard_demodulate(symbols_hat, frame.modulation)\n    return decided, bits\n\n\ndef _resolve_point_cfg(config: SystemConfig, section: str, mode: str) -> dict:\n    base = dict(config.raw[\"evaluation\"][section])\n    if mode == \"smoke\":\n        base.update(config.raw.get(\"smoke\", {}).get(\"evaluation\", {}).get(section, {}))\n    return base\n\n\ndef _curve_eval_nmse(\n    config: SystemConfig,\n    modulation: str,\n    realizations: int,\n    data_snr_db: list[float] | float,\n    pdr_db: list[float] | float,\n    backbone: torch.nn.Module | None,\n    adapters: dict[str, torch.nn.Module] | None,\n    x_name: str,\n    methods: list[str],\n    seed_offset: int,\n) -> pd.DataFrame:\n    x_values = data_snr_db if isinstance(data_snr_db, list) else pdr_db if isinstance(pdr_db, list) else [0.0]\n    rng = np.random.default_rng(config.seed + seed_offset)\n    rows: list[dict[str, float | str]] = []\n    for x_val in x_values:\n        print(f\"[eval:nmse] {x_name}={float(x_val):.3f} modulation={modulation} realizations={realizations}\", flush=True)\n        metric_samples: dict[str, list[float]] = {method: [] for method in methods}\n        for idx in range(realizations):\n            snr = float(x_val if x_name == \"data_snr_db\" else data_snr_db)\n            pdr = float(x_val if x_name == \"pdr_db\" else pdr_db)\n            frame = simulate_frame(config, modulation, snr, pdr, rng)\n            estimates = estimate_channels(frame, config, backbone=backbone, adapters=adapters)\n            for method in methods:\n                h_est = estimates[method]\n                metric_samples[method].append(nmse(h_est, frame.h_eff_support))\n            if (idx + 1) == 1 or (idx + 1) % 50 == 0 or (idx + 1) == realizations:\n                print(f\"[eval:nmse] {x_name}={float(x_val):.3f} progress={idx + 1}/{realizations}\", flush=True)\n        for method in methods:\n            values = np.array(metric_samples[method], dtype=float)\n            lo, hi = bootstrap_mean_ci(values, seed=config.seed + seed_offset + int(abs(hash((method, x_val))) % 1_000_000))\n            print(\n                f\"[eval:nmse] {x_name}={float(x_val):.3f} method={method} mean={float(np.mean(values)):.6e} \"\n                f\"ci=[{lo:.6e}, {hi:.6e}]\",\n                flush=True,\n            )\n            rows.append(\n                {\n                    x_name: float(x_val),\n                    \"method\": method,\n                    \"nmse\": float(np.mean(values)),\n                    \"ci_low\": lo,\n                    \"ci_high\": hi,\n                    \"realizations\": int(realizations),\n                    \"modulation\": modulation,\n                    \"effective_channel_method\": str(config.raw.get(\"numerics\", {}).get(\"effective_channel_method\", \"fast\")),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef _curve_eval_ber(\n    config: SystemConfig,\n    modulation: str,\n    data_snr_db: list[float] | float,\n    pdr_db: list[float] | float,\n    backbone: torch.nn.Module | None,\n    adapters: dict[str, torch.nn.Module] | None,\n    x_name: str,\n    methods: list[str],\n    seed_offset: int,\n    point_cfg: dict,\n    mode: str,\n) -> pd.DataFrame:\n    x_values = data_snr_db if isinstance(data_snr_db, list) else pdr_db if isinstance(pdr_db, list) else [0.0]\n    rng = np.random.default_rng(config.seed + seed_offset)\n    rows: list[dict[str, float | str | int]] = []\n    target_errors = int(point_cfg.get(\"target_bit_errors\", 200))\n    max_bits = int(point_cfg.get(\"max_bits\", 2_000_000))\n    min_frames = int(point_cfg.get(\"min_frames\", 1 if mode == \"smoke\" else 20))\n    solver_name = _default_solver_name(config, mode, point_cfg)\n    for x_val in x_values:\n        snr = float(x_val if x_name == \"data_snr_db\" else data_snr_db)\n        pdr = float(x_val if x_name == \"pdr_db\" else pdr_db)\n        print(\n            f\"[eval:ber] {x_name}={float(x_val):.3f} modulation={modulation} target_errors={target_errors} \"\n            f\"max_bits={max_bits} min_frames={min_frames} solver={solver_name}\",\n            flush=True,\n        )\n        err_counts = {method: 0 for method in methods}\n        total_bits = 0\n        frames = 0\n        while total_bits < max_bits and (frames < min_frames or any(err_counts[m] < target_errors for m in methods)):\n            frame = simulate_frame(config, modulation, snr, pdr, rng)\n            estimates = estimate_channels(frame, config, backbone=backbone, adapters=adapters)\n            for method in methods:\n                h_est = estimates[method]\n                _, bits_hat = detect_frame(frame, h_est, config, solver=solver_name)\n                err_counts[method] += int(np.count_nonzero(bits_hat != frame.bits))\n            total_bits += int(frame.bits.size)\n            frames += 1\n            if frames == 1 or frames % 10 == 0:\n                summary = \" \".join(f\"{m}={err_counts[m] / max(total_bits, 1):.3e}\" for m in methods)\n                print(f\"[eval:ber] {x_name}={float(x_val):.3f} frames={frames} bits={total_bits} {summary}\", flush=True)\n        for method in methods:\n            ber = err_counts[method] / max(total_bits, 1)\n            ci_low, ci_high = wilson_ci(err_counts[method], total_bits)\n            print(\n                f\"[eval:ber] {x_name}={float(x_val):.3f} method={method} ber={ber:.6e} \"\n                f\"errors={err_counts[method]} bits={total_bits} ci=[{ci_low:.6e}, {ci_high:.6e}]\",\n                flush=True,\n            )\n            rows.append(\n                {\n                    x_name: float(x_val),\n                    \"method\": method,\n                    \"ber\": float(ber),\n                    \"ci_low\": float(ci_low),\n                    \"ci_high\": float(ci_high),\n                    \"frames\": int(frames),\n                    \"bits\": int(total_bits),\n                    \"errors\": int(err_counts[method]),\n                    \"target_bit_errors\": int(target_errors),\n                    \"max_bits\": int(max_bits),\n                    \"modulation\": modulation,\n                    \"effective_channel_method\": str(config.raw.get(\"numerics\", {}).get(\"effective_channel_method\", \"fast\")),\n                    \"solver\": solver_name,\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef evaluate_nmse_vs_pdr(\n    config: SystemConfig,\n    backbone: torch.nn.Module | None,\n    adapters: dict[str, torch.nn.Module] | None,\n    mode: str,\n) -> pd.DataFrame:\n    cfg = _resolve_point_cfg(config, \"nmse_vs_pdr\", mode)\n    realizations = int(cfg.get(\"realizations\", cfg.get(\"frames\", 200)))\n    methods = list(cfg.get(\"methods\", [\"conventional\", \"cnn\", \"generic\", \"fb_lara\"]))\n    return _curve_eval_nmse(\n        config,\n        cfg[\"modulation\"],\n        realizations,\n        cfg[\"data_snr_db\"],\n        list(cfg[\"pdr_db\"]),\n        backbone,\n        adapters,\n        \"pdr_db\",\n        methods,\n        101,\n    )\n\n\ndef evaluate_nmse_vs_snr(\n    config: SystemConfig,\n    backbone: torch.nn.Module | None,\n    adapters: dict[str, torch.nn.Module] | None,\n    mode: str,\n) -> pd.DataFrame:\n    cfg = _resolve_point_cfg(config, \"nmse_vs_snr\", mode)\n    realizations = int(cfg.get(\"realizations\", cfg.get(\"frames\", 200)))\n    methods = list(cfg.get(\"methods\", [\"conventional\", \"cnn\", \"generic\", \"fb_lara\"]))\n    return _curve_eval_nmse(\n        config,\n        cfg[\"modulation\"],\n        realizations,\n        list(cfg[\"data_snr_db\"]),\n        cfg[\"pdr_db\"],\n        backbone,\n        adapters,\n        \"data_snr_db\",\n        methods,\n        102,\n    )\n\n\ndef evaluate_ber_vs_pdr(\n    config: SystemConfig,\n    backbone: torch.nn.Module | None,\n    adapters: dict[str, torch.nn.Module] | None,\n    mode: str,\n) -> pd.DataFrame:\n    cfg = _resolve_point_cfg(config, \"ber_vs_pdr\", mode)\n    methods = list(cfg.get(\"methods\", [\"conventional\", \"cnn\", \"generic\", \"fb_lara\", \"perfect\"]))\n    return _curve_eval_ber(\n        config,\n        cfg[\"modulation\"],\n        cfg[\"data_snr_db\"],\n        list(cfg[\"pdr_db\"]),\n        backbone,\n        adapters,\n        \"pdr_db\",\n        methods,\n        103,\n        cfg,\n        mode,\n    )\n\n\ndef evaluate_ber_vs_snr(\n    config: SystemConfig,\n    backbone: torch.nn.Module | None,\n    adapters: dict[str, torch.nn.Module] | None,\n    mode: str,\n) -> pd.DataFrame:\n    cfg = _resolve_point_cfg(config, \"ber_vs_snr\", mode)\n    frames_out = []\n    methods = list(cfg.get(\"methods\", [\"conventional\", \"cnn\", \"generic\", \"fb_lara\", \"perfect\"]))\n    for modulation in cfg[\"modulation\"]:\n        frames_out.append(\n            _curve_eval_ber(\n                config,\n                modulation,\n                list(cfg[\"data_snr_db\"]),\n                cfg[\"pdr_db\"],\n                backbone,\n                adapters,\n                \"data_snr_db\",\n                methods,\n                104 + len(frames_out),\n                cfg,\n                mode,\n            )\n        )\n    return pd.concat(frames_out, ignore_index=True)\n\n\ndef save_eval_outputs(df: pd.DataFrame, metric_name: str, prefix: str, config: SystemConfig) -> None:\n    out_dir = results_dir(config)\n    csv_path = out_dir / f\"{prefix}.csv\"\n    json_path = out_dir / f\"{prefix}.json\"\n    png_path = out_dir / f\"{prefix}.png\"\n    df.to_csv(csv_path, index=False)\n    save_json(json_path, {\"records\": df.to_dict(orient=\"records\")})\n    x_col = \"pdr_db\" if \"pdr_db\" in df.columns else \"data_snr_db\"\n    title = prefix.replace(\"_\", \" \").upper()\n    hue_col, style_col = curve_plot_columns(df)\n    save_curve_plot(df, x_col, metric_name, hue_col, title, png_path, logy=True, style_col=style_col)\n    print(f\"[eval] saved csv={csv_path}\", flush=True)\n    print(f\"[eval] saved json={json_path}\", flush=True)\n    print(f\"[eval] saved png={png_path}\", flush=True)\n\n\ndef compare_to_anchors(config: SystemConfig, nmse_pdr: pd.DataFrame, ber_pdr: pd.DataFrame, ber_snr: pd.DataFrame) -> dict:\n    anchors = config.raw[\"anchors\"]\n    summary = {}\n    conv_10 = float(nmse_pdr[(nmse_pdr[\"pdr_db\"] == 10.0) & (nmse_pdr[\"method\"] == \"conventional\")][\"nmse\"].iloc[0])\n    cnn_10 = float(nmse_pdr[(nmse_pdr[\"pdr_db\"] == 10.0) & (nmse_pdr[\"method\"] == \"cnn\")][\"nmse\"].iloc[0])\n    summary[\"nmse_vs_pdr\"] = {\n        \"paper\": anchors[\"nmse_vs_pdr\"][\"pdr_db_10\"],\n        \"reproduced\": {\"conventional\": conv_10, \"cnn\": cnn_10},\n    }\n    mask = (ber_pdr[\"pdr_db\"] == 5.0)\n    summary[\"ber_vs_pdr\"] = {\n        \"paper\": anchors[\"ber_vs_pdr\"][\"pdr_db_5_bpsk_snr_db_18\"],\n        \"reproduced\": {\n            method: float(ber_pdr[mask & (ber_pdr[\"method\"] == method)][\"ber\"].iloc[0]) for method in [\"conventional\", \"cnn\", \"perfect\"]\n        },\n    }\n    mask_bpsk = (ber_snr[\"data_snr_db\"] == 18.0) & (ber_snr[\"modulation\"] == \"bpsk\")\n    mask_qam8 = (ber_snr[\"data_snr_db\"] == 18.0) & (ber_snr[\"modulation\"] == \"8qam_cross\")\n    summary[\"ber_vs_snr_bpsk\"] = {\n        \"paper\": anchors[\"ber_vs_snr\"][\"bpsk_snr_db_18_pdr_db_5\"],\n        \"reproduced\": {\n            method: float(ber_snr[mask_bpsk & (ber_snr[\"method\"] == method)][\"ber\"].iloc[0]) for method in [\"conventional\", \"cnn\", \"perfect\"]\n        },\n    }\n    summary[\"ber_vs_snr_8qam\"] = {\n        \"paper\": anchors[\"ber_vs_snr\"][\"qam8_snr_db_18_pdr_db_5\"],\n        \"reproduced\": {\n            method: float(ber_snr[mask_qam8 & (ber_snr[\"method\"] == method)][\"ber\"].iloc[0]) for method in [\"conventional\", \"cnn\", \"perfect\"]\n        },\n    }\n    return summary\n",
  "src/zakotfs_novelty/lattice.py": "from __future__ import annotations\n\nimport numpy as np\n\nfrom .compat import dataclass_slots\nfrom .params import SystemConfig\nfrom .utils import centered_coord_to_index\n\n\n@dataclass_slots()\nclass SupportGeometry:\n    delta_k: int\n    delta_l: int\n    k_min: int\n    k_max: int\n    l_min: int\n    l_max: int\n    basis_delay: tuple[int, int]\n    basis_doppler: tuple[int, int]\n\n\ndef lattice_condition(k: int, l: int, config: SystemConfig) -> bool:\n    M, N, q = config.M, config.N, config.q\n    mod_inv = pow(2 * q, -1, M * N)\n    return ((2 * q * k - l) % M == 0) and ((k - l * (mod_inv - 2 * q)) % N == 0)\n\n\ndef enumerate_lattice_points(config: SystemConfig, radius_k: int = 64, radius_l: int = 96) -> list[tuple[int, int]]:\n    pts: list[tuple[int, int]] = []\n    for k in range(-radius_k, radius_k + 1):\n        for l in range(-radius_l, radius_l + 1):\n            if lattice_condition(k, l, config):\n                pts.append((k, l))\n    return pts\n\n\ndef derive_support_geometry(config: SystemConfig) -> SupportGeometry:\n    points = [pt for pt in enumerate_lattice_points(config) if pt != (0, 0)]\n    basis_doppler = min((pt for pt in points if pt[1] > 0), key=lambda pt: (abs(pt[0]), pt[1]))\n    independent = [pt for pt in points if (basis_doppler[0] * pt[1] - basis_doppler[1] * pt[0]) != 0]\n    basis_delay = min((pt for pt in independent if pt[0] > 0), key=lambda pt: (abs(pt[1]), pt[0]))\n    delta_k = abs(basis_delay[0])\n    delta_l = abs(basis_doppler[1])\n    k_max = delta_k // 2\n    l_max = delta_l // 2\n    return SupportGeometry(\n        delta_k=delta_k,\n        delta_l=delta_l,\n        k_min=-k_max,\n        k_max=k_max,\n        l_min=-l_max,\n        l_max=l_max,\n        basis_delay=basis_delay,\n        basis_doppler=basis_doppler,\n    )\n\n\ndef support_shape(config: SystemConfig) -> tuple[int, int]:\n    g = derive_support_geometry(config)\n    return g.delta_k, g.delta_l\n\n\ndef support_coords(config: SystemConfig) -> tuple[np.ndarray, np.ndarray]:\n    g = derive_support_geometry(config)\n    k = np.arange(g.k_min, g.k_max + 1, dtype=int)\n    l = np.arange(g.l_min, g.l_max + 1, dtype=int)\n    return np.meshgrid(k, l, indexing=\"ij\")\n\n\ndef support_mask(config: SystemConfig) -> np.ndarray:\n    mask = np.zeros((config.M, config.N), dtype=np.uint8)\n    g = derive_support_geometry(config)\n    for k in range(g.k_min, g.k_max + 1):\n        for l in range(g.l_min, g.l_max + 1):\n            mask[centered_coord_to_index(k, config.M), centered_coord_to_index(l, config.N)] = 1\n    return mask\n\n\ndef crop_support(arr: np.ndarray, config: SystemConfig) -> np.ndarray:\n    g = derive_support_geometry(config)\n    out = np.zeros((g.delta_k, g.delta_l), dtype=arr.dtype)\n    for i, k in enumerate(range(g.k_min, g.k_max + 1)):\n        for j, l in enumerate(range(g.l_min, g.l_max + 1)):\n            out[i, j] = arr[centered_coord_to_index(k, config.M), centered_coord_to_index(l, config.N)]\n    return out\n\n\ndef embed_support_image(support_img: np.ndarray, config: SystemConfig) -> np.ndarray:\n    full = np.zeros((config.M, config.N), dtype=support_img.dtype)\n    g = derive_support_geometry(config)\n    for i, k in enumerate(range(g.k_min, g.k_max + 1)):\n        for j, l in enumerate(range(g.l_min, g.l_max + 1)):\n            full[centered_coord_to_index(k, config.M), centered_coord_to_index(l, config.N)] = support_img[i, j]\n    return full\n",
  "src/zakotfs_novelty/metrics.py": "from __future__ import annotations\n\nimport numpy as np\n\n\ndef nmse(estimate: np.ndarray, truth: np.ndarray) -> float:\n    denom = float(np.sum(np.abs(truth) ** 2))\n    if denom == 0.0:\n        return 0.0\n    return float(np.sum(np.abs(estimate - truth) ** 2) / denom)\n\n\ndef ber_from_bits(bits_hat: np.ndarray, bits_true: np.ndarray) -> float:\n    return float(np.mean(bits_hat.reshape(-1) != bits_true.reshape(-1)))\n",
  "src/zakotfs_novelty/mmse.py": "from __future__ import annotations\n\nimport numpy as np\nimport torch\n\nfrom .operators import (\n    apply_heff_adjoint,\n    apply_heff_operator,\n    apply_support_adjoint,\n    apply_support_operator,\n    build_dense_heff_matrix,\n    build_dense_support_matrix,\n)\nfrom .params import SystemConfig\n\n\ndef _dense_matrix(channel_estimate: np.ndarray, config: SystemConfig) -> np.ndarray:\n    return (\n        build_dense_heff_matrix(channel_estimate, config)\n        if channel_estimate.shape == (config.M, config.N)\n        else build_dense_support_matrix(channel_estimate, config)\n    )\n\n\ndef mmse_ridge_lambda(noise_variance: float, E_d: float, config: SystemConfig) -> float:\n    data_variance = float(E_d) / float(config.Q)\n    if data_variance <= 0.0:\n        raise ValueError(\"E_d must be positive for MMSE detection\")\n    return float(noise_variance) / data_variance\n\n\ndef mmse_dense(\n    y: np.ndarray,\n    heff: np.ndarray,\n    noise_variance: float,\n    config: SystemConfig,\n    E_d: float = 1.0,\n) -> np.ndarray:\n    H = _dense_matrix(heff, config)\n    ridge = mmse_ridge_lambda(noise_variance, E_d, config)\n    A = H.conj().T @ H + ridge * np.eye(config.Q, dtype=np.complex64)\n    b = H.conj().T @ y.reshape(-1)\n    x = np.linalg.solve(A, b)\n    return x.reshape(config.M, config.N)\n\n\ndef mmse_dense_torch(\n    y: np.ndarray,\n    heff: np.ndarray,\n    noise_variance: float,\n    config: SystemConfig,\n    E_d: float = 1.0,\n    device: torch.device | str | None = None,\n) -> np.ndarray:\n    if device is None:\n        device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    else:\n        device = torch.device(device)\n    H = torch.from_numpy(_dense_matrix(heff, config)).to(device=device, dtype=torch.complex64)\n    y_vec = torch.from_numpy(y.reshape(-1)).to(device=device, dtype=torch.complex64)\n    ridge = mmse_ridge_lambda(noise_variance, E_d, config)\n    A = H.conj().transpose(0, 1) @ H + ridge * torch.eye(config.Q, device=device, dtype=torch.complex64)\n    b = H.conj().transpose(0, 1) @ y_vec\n    x = torch.linalg.solve(A, b)\n    return x.detach().cpu().numpy().reshape(config.M, config.N)\n\n\ndef cg_solve(operator, rhs: np.ndarray, tol: float, maxiter: int) -> np.ndarray:\n    x = np.zeros_like(rhs)\n    r = rhs - operator(x)\n    p = r.copy()\n    rs_old = np.vdot(r, r)\n    if not np.isfinite(rs_old) or np.real(rs_old) <= 0.0:\n        return x\n    for _ in range(maxiter):\n        Ap = operator(p)\n        denom = np.vdot(p, Ap)\n        if not np.isfinite(denom) or abs(denom) < 1e-20:\n            break\n        alpha = rs_old / denom\n        x = x + alpha * p\n        r = r - alpha * Ap\n        rs_new = np.vdot(r, r)\n        if not np.isfinite(rs_new):\n            break\n        if np.sqrt(np.real(rs_new)) < tol:\n            break\n        if abs(rs_old) < 1e-20:\n            break\n        p = r + (rs_new / rs_old) * p\n        rs_old = rs_new\n    return x\n\n\ndef mmse_iterative(\n    y: np.ndarray,\n    heff: np.ndarray,\n    noise_variance: float,\n    config: SystemConfig,\n    E_d: float = 1.0,\n) -> np.ndarray:\n    forward = apply_heff_operator if heff.shape == (config.M, config.N) else apply_support_operator\n    adjoint = apply_heff_adjoint if heff.shape == (config.M, config.N) else apply_support_adjoint\n    rhs = adjoint(heff, y, config)\n    ridge = mmse_ridge_lambda(noise_variance, E_d, config)\n\n    def normal_op(x_flat: np.ndarray) -> np.ndarray:\n        x = x_flat.reshape(config.M, config.N)\n        hx = forward(heff, x, config)\n        ahx = adjoint(heff, hx, config)\n        return (ahx + ridge * x).reshape(-1)\n\n    maxiter = int(config.detection[\"cg_maxiter\"])\n    x = cg_solve(normal_op, rhs.reshape(-1), float(config.detection[\"cg_tol\"]), maxiter)\n    return x.reshape(config.M, config.N)\n",
  "src/zakotfs_novelty/modulation.py": "from __future__ import annotations\n\nimport numpy as np\n\n\ndef constellation(name: str) -> tuple[np.ndarray, np.ndarray]:\n    name = name.lower().replace(\"-\", \"\").replace(\"_\", \"\")\n    if name == \"bpsk\":\n        points = np.array([-1.0, 1.0], dtype=np.complex64)\n        bits = np.array([[0], [1]], dtype=np.int8)\n        return points, bits\n    if name in {\"8qam\", \"8qamcross\"}:\n        points = np.array(\n            [\n                -3 + 0j,\n                -1 - 1j,\n                -1 + 1j,\n                0 - 3j,\n                0 + 3j,\n                1 - 1j,\n                1 + 1j,\n                3 + 0j,\n            ],\n            dtype=np.complex64,\n        )\n        points = points / np.sqrt(np.mean(np.abs(points) ** 2))\n        bits = np.array(\n            [\n                [0, 0, 0],\n                [0, 0, 1],\n                [0, 1, 1],\n                [0, 1, 0],\n                [1, 1, 0],\n                [1, 1, 1],\n                [1, 0, 1],\n                [1, 0, 0],\n            ],\n            dtype=np.int8,\n        )\n        return points, bits\n    if name in {\"8qamstar\"}:\n        angles = np.deg2rad(np.array([0, 45, 90, 135, 180, 225, 270, 315], dtype=float))\n        radii = np.array([1, np.sqrt(2), 1, np.sqrt(2), 1, np.sqrt(2), 1, np.sqrt(2)], dtype=float)\n        points = (radii * np.exp(1j * angles)).astype(np.complex64)\n        points = points / np.sqrt(np.mean(np.abs(points) ** 2))\n        bits = np.array([[i >> 2 & 1, i >> 1 & 1, i & 1] for i in range(8)], dtype=np.int8)\n        return points, bits\n    raise ValueError(f\"Unsupported modulation: {name}\")\n\n\ndef sample_symbols(modulation: str, shape: tuple[int, ...], rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:\n    points, bits = constellation(modulation)\n    indices = rng.integers(0, len(points), size=shape)\n    return points[indices], bits[indices]\n\n\ndef hard_demodulate(symbols: np.ndarray, modulation: str) -> tuple[np.ndarray, np.ndarray]:\n    points, bits = constellation(modulation)\n    distances = np.abs(symbols[..., None] - points[None, None, :]) ** 2\n    idx = np.argmin(distances, axis=-1)\n    return points[idx], bits[idx]\n",
  "src/zakotfs_novelty/operators.py": "from __future__ import annotations\n\nimport numpy as np\n\nfrom .compat import dataclass_slots\nfrom .lattice import derive_support_geometry\nfrom .params import SystemConfig\nfrom .utils import index_to_centered_coord\n\n\n@dataclass_slots()\nclass ShiftTerm:\n    delay: int\n    doppler: int\n    value: np.complex64\n\n\ndef nonzero_shift_terms(heff: np.ndarray, config: SystemConfig, tol: float = 1e-9) -> list[ShiftTerm]:\n    terms: list[ShiftTerm] = []\n    for i in range(config.M):\n        for j in range(config.N):\n            value = heff[i, j]\n            if abs(value) > tol:\n                terms.append(ShiftTerm(index_to_centered_coord(i, config.M), index_to_centered_coord(j, config.N), np.complex64(value)))\n    return terms\n\n\ndef support_shift_terms(support_img: np.ndarray, config: SystemConfig, tol: float = 1e-9) -> list[ShiftTerm]:\n    g = derive_support_geometry(config)\n    terms: list[ShiftTerm] = []\n    for i, k in enumerate(range(g.k_min, g.k_max + 1)):\n        for j, l in enumerate(range(g.l_min, g.l_max + 1)):\n            value = support_img[i, j]\n            if abs(value) > tol:\n                terms.append(ShiftTerm(k, l, np.complex64(value)))\n    return terms\n\n\ndef apply_shift_terms(terms: list[ShiftTerm], x: np.ndarray, config: SystemConfig) -> np.ndarray:\n    M, N, Q = config.M, config.N, config.Q\n    k = np.arange(M, dtype=int)[:, None]\n    l = np.arange(N, dtype=int)[None, :]\n    y = np.zeros((M, N), dtype=np.complex64)\n    for term in terms:\n        src_k = k - term.delay\n        src_l = l - term.doppler\n        base_k = np.mod(src_k, M)\n        base_l = np.mod(src_l, N)\n        wrap_k = np.floor_divide(src_k - base_k, M)\n        x_qp = x[base_k, base_l] * np.exp(1j * 2 * np.pi * wrap_k * base_l / N)\n        phase = np.exp(1j * 2 * np.pi * src_k * term.doppler / Q)\n        y += term.value * x_qp * phase\n    return y\n\n\ndef apply_heff_operator(heff: np.ndarray, x: np.ndarray, config: SystemConfig) -> np.ndarray:\n    return apply_shift_terms(nonzero_shift_terms(heff, config), x, config)\n\n\ndef apply_support_operator(support_img: np.ndarray, x: np.ndarray, config: SystemConfig) -> np.ndarray:\n    return apply_shift_terms(support_shift_terms(support_img, config), x, config)\n\n\ndef apply_shift_terms_adjoint(terms: list[ShiftTerm], y: np.ndarray, config: SystemConfig) -> np.ndarray:\n    M, N, Q = config.M, config.N, config.Q\n    k = np.arange(M, dtype=int)[:, None]\n    l = np.arange(N, dtype=int)[None, :]\n    x = np.zeros((M, N), dtype=np.complex64)\n    for term in terms:\n        src_k = k - term.delay\n        src_l = l - term.doppler\n        base_k = np.broadcast_to(np.mod(src_k, M), (M, N))\n        base_l = np.broadcast_to(np.mod(src_l, N), (M, N))\n        wrap_k = np.floor_divide(src_k - base_k, M)\n        phase_qp = np.exp(1j * 2 * np.pi * wrap_k * base_l / N)\n        phase = np.exp(1j * 2 * np.pi * src_k * term.doppler / Q)\n        coeff = np.conjugate(term.value * phase_qp * phase)\n        np.add.at(x, (base_k.ravel(), base_l.ravel()), (coeff * y).ravel())\n    return x\n\n\ndef apply_heff_adjoint(heff: np.ndarray, y: np.ndarray, config: SystemConfig) -> np.ndarray:\n    return apply_shift_terms_adjoint(nonzero_shift_terms(heff, config), y, config)\n\n\ndef apply_support_adjoint(support_img: np.ndarray, y: np.ndarray, config: SystemConfig) -> np.ndarray:\n    return apply_shift_terms_adjoint(support_shift_terms(support_img, config), y, config)\n\n\ndef build_dense_from_terms(terms: list[ShiftTerm], config: SystemConfig) -> np.ndarray:\n    M, N, Q = config.M, config.N, config.Q\n    rows = np.arange(Q, dtype=int).reshape(M, N)\n    k = np.arange(M, dtype=int)[:, None]\n    l = np.arange(N, dtype=int)[None, :]\n    H = np.zeros((Q, Q), dtype=np.complex64)\n    for term in terms:\n        src_k = k - term.delay\n        src_l = l - term.doppler\n        base_k = np.mod(src_k, M)\n        base_l = np.mod(src_l, N)\n        wrap_k = np.floor_divide(src_k - base_k, M)\n        cols = base_k * N + base_l\n        coeff = term.value * np.exp(1j * 2 * np.pi * wrap_k * base_l / N) * np.exp(1j * 2 * np.pi * src_k * term.doppler / Q)\n        H[rows.ravel(), cols.ravel()] += coeff.ravel()\n    return H\n\n\ndef build_dense_heff_matrix(heff: np.ndarray, config: SystemConfig) -> np.ndarray:\n    return build_dense_from_terms(nonzero_shift_terms(heff, config), config)\n\n\ndef build_dense_support_matrix(support_img: np.ndarray, config: SystemConfig) -> np.ndarray:\n    return build_dense_from_terms(support_shift_terms(support_img, config), config)\n\n\ndef dense_apply(heff: np.ndarray, x: np.ndarray, config: SystemConfig) -> np.ndarray:\n    H = build_dense_heff_matrix(heff, config)\n    y = H @ x.reshape(-1)\n    return y.reshape(config.M, config.N)\n",
  "src/zakotfs_novelty/params.py": "from __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any\n\nimport yaml\n\nfrom .compat import dataclass_slots\n\n\ndef _deep_update(base: dict[str, Any], override: dict[str, Any]) -> dict[str, Any]:\n    merged = dict(base)\n    for key, value in override.items():\n        if isinstance(value, dict) and isinstance(merged.get(key), dict):\n            merged[key] = _deep_update(merged[key], value)\n        else:\n            merged[key] = value\n    return merged\n\n\ndef _load_yaml(path: Path) -> dict[str, Any]:\n    with path.open(\"r\", encoding=\"utf-8\") as handle:\n        data = yaml.safe_load(handle) or {}\n    inherits = data.pop(\"inherits\", None)\n    if inherits:\n        parent = _load_yaml((path.parent / inherits).resolve() if not Path(inherits).is_absolute() else Path(inherits))\n        return _deep_update(parent, data)\n    return data\n\n\n@dataclass_slots()\nclass SystemConfig:\n    raw: dict[str, Any]\n    root: Path\n\n    @property\n    def frame(self) -> dict[str, Any]:\n        return self.raw[\"frame\"]\n\n    @property\n    def pulse(self) -> dict[str, Any]:\n        return self.raw[\"pulse\"]\n\n    @property\n    def channel(self) -> dict[str, Any]:\n        return self.raw[\"channel\"]\n\n    @property\n    def estimation(self) -> dict[str, Any]:\n        return self.raw[\"estimation\"]\n\n    @property\n    def detection(self) -> dict[str, Any]:\n        return self.raw[\"detection\"]\n\n    @property\n    def seed(self) -> int:\n        return int(self.raw[\"seed\"])\n\n    @property\n    def M(self) -> int:\n        return int(self.frame[\"M\"])\n\n    @property\n    def N(self) -> int:\n        return int(self.frame[\"N\"])\n\n    @property\n    def Q(self) -> int:\n        return self.M * self.N\n\n    @property\n    def tau_p(self) -> float:\n        return float(self.frame[\"tau_p_s\"])\n\n    @property\n    def nu_p(self) -> float:\n        return float(self.frame[\"nu_p_hz\"])\n\n    @property\n    def T(self) -> float:\n        return float(self.frame[\"T_s\"])\n\n    @property\n    def B(self) -> float:\n        return float(self.frame[\"B_hz\"])\n\n    @property\n    def q(self) -> int:\n        return int(self.frame[\"q\"])\n\n\ndef load_config(path: str | Path) -> SystemConfig:\n    resolved = Path(path).resolve()\n    return SystemConfig(raw=_load_yaml(resolved), root=resolved.parent.parent)\n",
  "src/zakotfs_novelty/plotting.py": "from __future__ import annotations\n\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport seaborn as sns\n\nfrom .compat import strict_zip\n\n\ndef curve_plot_columns(df: pd.DataFrame) -> tuple[str, str | None]:\n    style_col = None\n    if \"modulation\" in df.columns and df[\"modulation\"].nunique() > 1:\n        style_col = \"modulation\"\n    return \"method\", style_col\n\n\ndef save_curve_plot(\n    df: pd.DataFrame,\n    x_col: str,\n    y_col: str,\n    hue_col: str,\n    title: str,\n    path: Path,\n    logy: bool = True,\n    style_col: str | None = None,\n) -> None:\n    plt.figure(figsize=(7, 4.5))\n    plot_kwargs = {\"data\": df, \"x\": x_col, \"y\": y_col, \"hue\": hue_col}\n    if style_col is None:\n        plot_kwargs[\"marker\"] = \"o\"\n    else:\n        plot_kwargs[\"style\"] = style_col\n        plot_kwargs[\"markers\"] = True\n        plot_kwargs[\"dashes\"] = True\n    sns.lineplot(**plot_kwargs)\n    if logy:\n        plt.yscale(\"log\")\n    plt.title(title)\n    plt.grid(True, alpha=0.25)\n    plt.tight_layout()\n    plt.savefig(path, dpi=180)\n    plt.close()\n\n\ndef save_heatmaps(images: list[np.ndarray], titles: list[str], path: Path) -> None:\n    fig, axes = plt.subplots(1, len(images), figsize=(4.2 * len(images), 4))\n    if len(images) == 1:\n        axes = [axes]\n    for ax, image, title in strict_zip(axes, images, titles):\n        im = ax.imshow(np.abs(image), origin=\"lower\", aspect=\"auto\", cmap=\"magma\")\n        ax.set_title(title)\n        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n    fig.tight_layout()\n    fig.savefig(path, dpi=180)\n    plt.close(fig)\n",
  "src/zakotfs_novelty/pulses.py": "from __future__ import annotations\n\nfrom functools import lru_cache\n\nimport numpy as np\n\nfrom .params import SystemConfig\nfrom .utils import centered_mesh\n\n\ndef gs_delay_pulse(delay_s: np.ndarray, config: SystemConfig) -> np.ndarray:\n    p = config.pulse\n    B = config.B\n    value = (\n        p[\"Omega_tau\"]\n        * np.sqrt(B)\n        * np.sinc(B * delay_s)\n        * np.exp(-p[\"alpha_tau\"] * (B * delay_s) ** 2)\n    )\n    return np.asarray(value, dtype=np.complex64)\n\n\ndef gs_doppler_pulse(doppler_hz: np.ndarray, config: SystemConfig) -> np.ndarray:\n    p = config.pulse\n    T = config.T\n    value = (\n        p[\"Omega_nu\"]\n        * np.sqrt(T)\n        * np.sinc(T * doppler_hz)\n        * np.exp(-p[\"alpha_nu\"] * (T * doppler_hz) ** 2)\n    )\n    return np.asarray(value, dtype=np.complex64)\n\n\ndef gs_transmit_pulse(delay_s: np.ndarray, doppler_hz: np.ndarray, config: SystemConfig) -> np.ndarray:\n    return (gs_delay_pulse(delay_s, config) * gs_doppler_pulse(doppler_hz, config)).astype(np.complex64)\n\n\ndef gs_matched_filter(delay_s: np.ndarray, doppler_hz: np.ndarray, config: SystemConfig) -> np.ndarray:\n    return np.conjugate(gs_transmit_pulse(-delay_s, -doppler_hz, config)) * np.exp(1j * 2 * np.pi * delay_s * doppler_hz)\n\n\ndef _quad_points(config: SystemConfig, axis: str) -> int:\n    pulse_cfg = config.raw.get(\"numerics\", {}).get(\"pulse_quadrature\", {})\n    return int(pulse_cfg.get(f\"{axis}_points\", 2049))\n\n\ndef _quad_span(config: SystemConfig, axis: str) -> float:\n    pulse_cfg = config.raw.get(\"numerics\", {}).get(\"pulse_quadrature\", {})\n    return float(pulse_cfg.get(f\"{axis}_span\", 10.0))\n\n\n@lru_cache(maxsize=8)\ndef _delay_quadrature_grid(B: float, points: int, span: float) -> np.ndarray:\n    return np.linspace(-span / B, span / B, points, dtype=float)\n\n\n@lru_cache(maxsize=8)\ndef _doppler_quadrature_grid(T: float, points: int, span: float) -> np.ndarray:\n    return np.linspace(-span / T, span / T, points, dtype=float)\n\n\ndef delay_quadrature_grid(config: SystemConfig) -> np.ndarray:\n    return _delay_quadrature_grid(config.B, _quad_points(config, \"delay\"), _quad_span(config, \"delay\"))\n\n\ndef doppler_quadrature_grid(config: SystemConfig) -> np.ndarray:\n    return _doppler_quadrature_grid(config.T, _quad_points(config, \"doppler\"), _quad_span(config, \"doppler\"))\n\n\ndef gs_delay_autocorrelation(delay_offset_s: np.ndarray, config: SystemConfig) -> np.ndarray:\n    tau = delay_quadrature_grid(config)\n    w = gs_delay_pulse(tau, config).astype(np.complex128)\n    shifted = gs_delay_pulse(np.asarray(delay_offset_s)[..., None] - tau[None, :], config).astype(np.complex128)\n    return np.trapezoid(w[None, :] * shifted, tau, axis=-1).astype(np.complex64)\n\n\ndef gs_doppler_autocorrelation(doppler_offset_hz: np.ndarray, config: SystemConfig) -> np.ndarray:\n    nu = doppler_quadrature_grid(config)\n    w = gs_doppler_pulse(nu, config).astype(np.complex128)\n    shifted = gs_doppler_pulse(np.asarray(doppler_offset_hz)[..., None] - nu[None, :], config).astype(np.complex128)\n    return np.trapezoid(w[None, :] * shifted, nu, axis=-1).astype(np.complex64)\n\n\ndef gs_delay_overlap_exact(delay_s: np.ndarray, path_delay_s: float, path_doppler_hz: float, config: SystemConfig) -> np.ndarray:\n    tau = delay_quadrature_grid(config)\n    w = gs_delay_pulse(tau, config).astype(np.complex128)\n    shifted = gs_delay_pulse(np.asarray(delay_s)[..., None] - path_delay_s - tau[None, :], config).astype(np.complex128)\n    phase = np.exp(-1j * 2 * np.pi * path_doppler_hz * tau)[None, :]\n    return np.trapezoid(w[None, :] * shifted * phase, tau, axis=-1).astype(np.complex64)\n\n\ndef gs_doppler_overlap_exact(delay_s: np.ndarray, doppler_hz: np.ndarray, path_doppler_hz: float, config: SystemConfig) -> np.ndarray:\n    nu = doppler_quadrature_grid(config)\n    w = gs_doppler_pulse(nu, config).astype(np.complex128)\n    phase = np.exp(1j * 2 * np.pi * np.asarray(delay_s)[:, None] * nu[None, :]).astype(np.complex128)\n    shifted = gs_doppler_pulse(np.asarray(doppler_hz)[:, None] - path_doppler_hz - nu[None, :], config).astype(np.complex128)\n    integrand = phase[:, None, :] * shifted[None, :, :] * w[None, None, :]\n    return np.trapezoid(integrand, nu, axis=-1).astype(np.complex64)\n\n\ndef effective_pulse_kernel(delay_s: np.ndarray, doppler_hz: np.ndarray, config: SystemConfig) -> np.ndarray:\n    # Kept as a compact localized envelope helper. The channel code now builds the\n    # paper-consistent phase outside this helper.\n    p = config.pulse\n    B = config.B\n    T = config.T\n    envelope = (\n        p[\"Omega_tau\"]\n        * p[\"Omega_nu\"]\n        * np.sinc(B * delay_s)\n        * np.sinc(T * doppler_hz)\n        * np.exp(-p[\"alpha_tau\"] * (B * delay_s) ** 2 - p[\"alpha_nu\"] * (T * doppler_hz) ** 2)\n    )\n    return np.asarray(envelope, dtype=np.complex64)\n\n\ndef chirp_spreading_filter(config: SystemConfig) -> np.ndarray:\n    M, N, q = config.M, config.N, config.q\n    k = np.arange(M, dtype=float)[:, None]\n    l = np.arange(N, dtype=float)[None, :]\n    return (np.exp(1j * 2 * np.pi * q * (k**2 + l**2) / (M * N)) / np.sqrt(M * N)).astype(np.complex64)\n\n\ndef mn_periodic_chirp_filter(config: SystemConfig) -> np.ndarray:\n    Q = config.Q\n    q = config.q\n    k = np.arange(Q, dtype=float)[:, None]\n    l = np.arange(Q, dtype=float)[None, :]\n    return (np.exp(1j * 2 * np.pi * q * (k**2 + l**2) / Q) / Q).astype(np.complex64)\n\n\ndef support_kernel_grid(config: SystemConfig) -> tuple[np.ndarray, np.ndarray]:\n    k, l = centered_mesh(config)\n    delay_s = k / config.B\n    doppler_hz = l / config.T\n    return delay_s, doppler_hz\n",
  "src/zakotfs_novelty/training.py": "from __future__ import annotations\n\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom .adapter import AdapterFeatureDataset\nfrom .cnn_model import GenericResidualAdapter, LatticeAliasAdapter, PaperCNN, ResidualAdapter\nfrom .dataset import simulate_frame\nfrom .params import SystemConfig, load_config\nfrom .utils import load_json, logs_dir, pin_memory_enabled, resolve_torch_device, save_json, set_global_seed\n\n\nclass ComplexSupportDataset(Dataset):\n    def __init__(self, path: Path) -> None:\n        self.path = Path(path)\n        default_cfg = Path(\"novelty_paper/configs/train.yaml\")\n        if not default_cfg.exists():\n            default_cfg = Path(\"configs/train.yaml\")\n        self.config = load_config(default_cfg) if self.path.suffix == \".json\" else None\n        self.cache_in_ram = bool(self.config.raw.get(\"dataset\", {}).get(\"cache_in_ram\", False)) if self.config is not None else False\n        if self.path.suffix == \".npz\":\n            loaded = np.load(path)\n            self.kind = \"npz\"\n            self.inputs = loaded[\"inputs\"].astype(np.complex64)\n            self.targets = loaded[\"targets\"].astype(np.complex64)\n            self.meta = {}\n        elif self.path.suffix == \".json\":\n            self.meta = load_json(self.path)\n            generator = str(self.meta.get(\"generator\", \"on_the_fly\"))\n            if generator == \"memmap_fp16\":\n                self.kind = \"memmap_fp16\"\n                load_mode = None if self.cache_in_ram else \"r\"\n                self.inputs_re = np.load(self._resolve_meta_path(\"inputs_re_path\"), mmap_mode=load_mode)\n                self.inputs_im = np.load(self._resolve_meta_path(\"inputs_im_path\"), mmap_mode=load_mode)\n                self.targets_re = np.load(self._resolve_meta_path(\"targets_re_path\"), mmap_mode=load_mode)\n                self.targets_im = np.load(self._resolve_meta_path(\"targets_im_path\"), mmap_mode=load_mode)\n                if self.cache_in_ram:\n                    print(f\"[train] caching dataset into RAM from {self.path}\", flush=True)\n                    self.inputs_re = np.asarray(self.inputs_re)\n                    self.inputs_im = np.asarray(self.inputs_im)\n                    self.targets_re = np.asarray(self.targets_re)\n                    self.targets_im = np.asarray(self.targets_im)\n                self.inputs = None\n                self.targets = None\n            else:\n                self.kind = \"generated\"\n                self.inputs = None\n                self.targets = None\n        else:\n            raise ValueError(f\"Unsupported dataset path: {path}\")\n\n    def _resolve_meta_path(self, key: str) -> str:\n        target = Path(str(self.meta[key]))\n        if target.is_absolute():\n            return str(target)\n        return str((self.path.parent / target).resolve())\n\n    def __len__(self) -> int:\n        return int(self.inputs.shape[0] if self.kind == \"npz\" else self.meta[\"size\"])\n\n    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:\n        if self.kind == \"npz\":\n            return torch.from_numpy(self.inputs[index]).to(torch.complex64), torch.from_numpy(self.targets[index]).to(torch.complex64)\n        if self.kind == \"memmap_fp16\":\n            x_re = torch.from_numpy(np.asarray(self.inputs_re[index], dtype=np.float32))\n            x_im = torch.from_numpy(np.asarray(self.inputs_im[index], dtype=np.float32))\n            y_re = torch.from_numpy(np.asarray(self.targets_re[index], dtype=np.float32))\n            y_im = torch.from_numpy(np.asarray(self.targets_im[index], dtype=np.float32))\n            return torch.complex(x_re, x_im), torch.complex(y_re, y_im)\n        pdrs = list(map(float, self.meta[\"pdr_db\"]))\n        per_pdr = int(self.meta[\"per_pdr\"])\n        pdr_idx = min(index // per_pdr, len(pdrs) - 1)\n        pdr_db = pdrs[pdr_idx]\n        sample_seed = int(self.meta[\"seed\"]) + int(index)\n        frame = simulate_frame(self.config, str(self.meta.get(\"modulation\", \"bpsk\")), float(self.meta[\"snr_db\"]), pdr_db, np.random.default_rng(sample_seed))\n        return torch.from_numpy(frame.support_input).to(torch.complex64), torch.from_numpy(frame.support_true).to(torch.complex64)\n\n\ndef _loss_fn(model: PaperCNN, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:\n    x_re = x.real.unsqueeze(1).to(dtype=torch.float32)\n    x_im = x.imag.unsqueeze(1).to(dtype=torch.float32)\n    target_re = y.real.unsqueeze(1).to(dtype=torch.float32)\n    target_im = y.imag.unsqueeze(1).to(dtype=torch.float32)\n    pred_re = model(x_re)\n    pred_im = model(x_im)\n    eps = 1.0e-12\n    num = ((pred_re - target_re) ** 2).sum(dim=(1, 2, 3)) + ((pred_im - target_im) ** 2).sum(dim=(1, 2, 3))\n    den = (target_re**2).sum(dim=(1, 2, 3)) + (target_im**2).sum(dim=(1, 2, 3))\n    return (num / den.clamp_min(eps)).mean()\n\n\ndef train_cnn(config: SystemConfig, train_path: Path, val_path: Path, mode: str = \"full\") -> Path:\n    set_global_seed(config.seed)\n    if mode == \"smoke\":\n        training_cfg = {**config.raw[\"training\"], **config.raw.get(\"smoke\", {}).get(\"training\", {})}\n    else:\n        training_cfg = config.raw[\"training\"]\n    device = resolve_torch_device(config)\n    model = PaperCNN().to(device=device, dtype=torch.float32)\n    train_ds = ComplexSupportDataset(train_path)\n    val_ds = ComplexSupportDataset(val_path)\n    num_workers = int(config.raw.get(\"dataset\", {}).get(\"num_workers\", 0))\n    print(\n        f\"[train] mode={mode} device={device} batch_size={training_cfg['batch_size']} epochs={training_cfg['epochs']} \"\n        f\"train_samples={len(train_ds)} val_samples={len(val_ds)} num_workers={num_workers}\",\n        flush=True,\n    )\n    print(f\"[train] train_source={train_path} val_source={val_path}\", flush=True)\n    train_loader = DataLoader(\n        train_ds,\n        batch_size=int(training_cfg[\"batch_size\"]),\n        shuffle=True,\n        num_workers=num_workers,\n        pin_memory=pin_memory_enabled(device),\n        persistent_workers=num_workers > 0,\n    )\n    val_loader = DataLoader(\n        val_ds,\n        batch_size=int(training_cfg[\"batch_size\"]),\n        shuffle=False,\n        num_workers=num_workers,\n        pin_memory=pin_memory_enabled(device),\n        persistent_workers=num_workers > 0,\n    )\n    optimizer = torch.optim.Adam(model.parameters(), lr=float(training_cfg[\"learning_rate\"]))\n    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(\n        optimizer,\n        factor=float(training_cfg[\"scheduler\"][\"factor\"]),\n        patience=int(training_cfg[\"scheduler\"][\"patience\"]),\n        min_lr=float(training_cfg[\"scheduler\"][\"min_lr\"]),\n    )\n    history: list[dict[str, float]] = []\n    best_val = float(\"inf\")\n    ckpt_dir = logs_dir(config) / \"checkpoints\"\n    ckpt_dir.mkdir(parents=True, exist_ok=True)\n    ckpt_path = ckpt_dir / f\"{mode}_{training_cfg['checkpoint_name']}\"\n    latest_ckpt_path = ckpt_dir / f\"{mode}_cnn_latest.pt\"\n    start = time.time()\n    start_epoch = 0\n    if bool(training_cfg.get(\"auto_resume\", True)) and latest_ckpt_path.exists():\n        payload = torch.load(latest_ckpt_path, map_location=device)\n        model.load_state_dict(payload[\"state_dict\"])\n        optimizer.load_state_dict(payload[\"optimizer_state\"])\n        scheduler.load_state_dict(payload[\"scheduler_state\"])\n        history = list(payload.get(\"history\", []))\n        best_val = float(payload.get(\"best_val_loss\", float(\"inf\")))\n        start_epoch = int(payload.get(\"epoch\", 0))\n        start = time.time() - float(payload.get(\"elapsed_s\", 0.0))\n        print(\n            f\"[train] auto-resume from {latest_ckpt_path} starting at epoch {start_epoch + 1}/{int(training_cfg['epochs'])}\",\n            flush=True,\n        )\n    for epoch in range(start_epoch, int(training_cfg[\"epochs\"])):\n        print(f\"[train] epoch {epoch + 1}/{int(training_cfg['epochs'])} started\", flush=True)\n        model.train()\n        train_losses = []\n        num_train_batches = len(train_loader)\n        for batch_idx, (x, y) in enumerate(train_loader, start=1):\n            x = x.to(device=device, dtype=torch.complex64)\n            y = y.to(device=device, dtype=torch.complex64)\n            optimizer.zero_grad(set_to_none=True)\n            loss = _loss_fn(model, x, y)\n            loss.backward()\n            optimizer.step()\n            train_losses.append(float(loss.detach().cpu()))\n            if batch_idx == 1 or batch_idx % 100 == 0 or batch_idx == num_train_batches:\n                print(\n                    f\"[train] epoch {epoch + 1} batch {batch_idx}/{num_train_batches} loss={train_losses[-1]:.6f}\",\n                    flush=True,\n                )\n        model.eval()\n        val_losses = []\n        num_val_batches = len(val_loader)\n        with torch.no_grad():\n            for batch_idx, (x, y) in enumerate(val_loader, start=1):\n                x = x.to(device=device, dtype=torch.complex64)\n                y = y.to(device=device, dtype=torch.complex64)\n                val_losses.append(float(_loss_fn(model, x, y).detach().cpu()))\n                if batch_idx == 1 or batch_idx % 100 == 0 or batch_idx == num_val_batches:\n                    print(\n                        f\"[train] epoch {epoch + 1} val_batch {batch_idx}/{num_val_batches} loss={val_losses[-1]:.6f}\",\n                        flush=True,\n                    )\n        train_loss = float(np.mean(train_losses))\n        val_loss = float(np.mean(val_losses))\n        scheduler.step(val_loss)\n        record = {\n            \"epoch\": epoch + 1,\n            \"train_loss\": train_loss,\n            \"val_loss\": val_loss,\n            \"lr\": float(optimizer.param_groups[0][\"lr\"]),\n            \"elapsed_s\": float(time.time() - start),\n        }\n        history.append(record)\n        print(\n            f\"[train] epoch {epoch + 1} done train_loss={train_loss:.6f} val_loss={val_loss:.6f} \"\n            f\"lr={record['lr']:.6e} elapsed_s={record['elapsed_s']:.1f}\",\n            flush=True,\n        )\n        improved = val_loss < best_val\n        if improved:\n            best_val = val_loss\n        latest_payload = {\n            \"epoch\": epoch + 1,\n            \"state_dict\": model.state_dict(),\n            \"optimizer_state\": optimizer.state_dict(),\n            \"scheduler_state\": scheduler.state_dict(),\n            \"history\": history,\n            \"best_val_loss\": best_val,\n            \"elapsed_s\": record[\"elapsed_s\"],\n        }\n        torch.save(latest_payload, latest_ckpt_path)\n        print(f\"[train] latest checkpoint saved to {latest_ckpt_path}\", flush=True)\n        if bool(training_cfg.get(\"save_epoch_checkpoints\", True)):\n            epoch_ckpt_path = ckpt_dir / f\"{mode}_epoch_{epoch + 1:03d}.pt\"\n            torch.save(latest_payload, epoch_ckpt_path)\n            print(f\"[train] epoch checkpoint saved to {epoch_ckpt_path}\", flush=True)\n        if improved:\n            torch.save(latest_payload, ckpt_path)\n            print(f\"[train] new best checkpoint saved to {ckpt_path} with val_loss={best_val:.6f}\", flush=True)\n    save_json(logs_dir(config) / f\"train_history_{mode}.json\", {\"history\": history, \"best_val_loss\": best_val})\n    print(f\"[train] finished best_val_loss={best_val:.6f} history={logs_dir(config) / f'train_history_{mode}.json'}\", flush=True)\n    return ckpt_path\n\n\ndef load_cnn_checkpoint(config: SystemConfig, checkpoint_path: Path | None = None) -> PaperCNN:\n    if checkpoint_path is None:\n        checkpoint_path = logs_dir(config) / \"checkpoints\" / \"full_cnn_best.pt\"\n        if not checkpoint_path.exists():\n            checkpoint_path = logs_dir(config) / \"checkpoints\" / \"smoke_cnn_best.pt\"\n    device = resolve_torch_device(config)\n    model = PaperCNN().to(device=device, dtype=torch.float32)\n    payload = torch.load(checkpoint_path, map_location=device)\n    model.load_state_dict(payload[\"state_dict\"])\n    model.eval()\n    print(f\"[train] loaded checkpoint {checkpoint_path} on device={device}\", flush=True)\n    return model\n\n\ndef instantiate_adapter(adapter_kind: str) -> ResidualAdapter:\n    kind = str(adapter_kind).lower()\n    if kind == \"generic\":\n        return GenericResidualAdapter()\n    if kind == \"fb_lara\":\n        return LatticeAliasAdapter()\n    raise ValueError(f\"Unknown adapter kind '{adapter_kind}'\")\n\n\ndef _adapter_loss_fn(adapter: ResidualAdapter, x: torch.Tensor, residual_target: torch.Tensor, residual_weight: float) -> torch.Tensor:\n    pred = adapter(x)\n    h_base = x[:, 2:4]\n    h_true = h_base + residual_target\n    eps = 1.0e-12\n    err_num = ((pred - residual_target) ** 2).sum(dim=(1, 2, 3))\n    delta_num = (pred**2).sum(dim=(1, 2, 3))\n    denom = (h_true**2).sum(dim=(1, 2, 3)).clamp_min(eps)\n    return ((err_num / denom) + residual_weight * (delta_num / denom)).mean()\n\n\ndef train_adapter(\n    config: SystemConfig,\n    train_path: Path,\n    val_path: Path,\n    adapter_kind: str,\n    mode: str = \"full\",\n) -> Path:\n    set_global_seed(config.seed)\n    if mode == \"smoke\":\n        training_cfg = {**config.raw[\"adapter_training\"], **config.raw.get(\"smoke\", {}).get(\"adapter_training\", {})}\n    else:\n        training_cfg = config.raw[\"adapter_training\"]\n    device = resolve_torch_device(config)\n    adapter = instantiate_adapter(adapter_kind).to(device=device, dtype=torch.float32)\n    train_ds = AdapterFeatureDataset(train_path, adapter_kind=adapter_kind)\n    val_ds = AdapterFeatureDataset(val_path, adapter_kind=adapter_kind)\n    num_workers = int(config.raw.get(\"adapter_dataset\", {}).get(\"num_workers\", 0))\n    print(\n        f\"[adapter-train] mode={mode} kind={adapter_kind} device={device} batch_size={training_cfg['batch_size']} \"\n        f\"epochs={training_cfg['epochs']} train_samples={len(train_ds)} val_samples={len(val_ds)} num_workers={num_workers}\",\n        flush=True,\n    )\n    train_loader = DataLoader(\n        train_ds,\n        batch_size=int(training_cfg[\"batch_size\"]),\n        shuffle=True,\n        num_workers=num_workers,\n        pin_memory=pin_memory_enabled(device),\n        persistent_workers=num_workers > 0,\n    )\n    val_loader = DataLoader(\n        val_ds,\n        batch_size=int(training_cfg[\"batch_size\"]),\n        shuffle=False,\n        num_workers=num_workers,\n        pin_memory=pin_memory_enabled(device),\n        persistent_workers=num_workers > 0,\n    )\n    optimizer = torch.optim.Adam(adapter.parameters(), lr=float(training_cfg[\"learning_rate\"]))\n    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(\n        optimizer,\n        factor=float(training_cfg[\"scheduler\"][\"factor\"]),\n        patience=int(training_cfg[\"scheduler\"][\"patience\"]),\n        min_lr=float(training_cfg[\"scheduler\"][\"min_lr\"]),\n    )\n    residual_weight = float(training_cfg.get(\"residual_penalty\", 1.0e-3))\n    history: list[dict[str, float]] = []\n    best_val = float(\"inf\")\n    epochs_without_improvement = 0\n    early_stop_patience = int(training_cfg.get(\"early_stop_patience\", training_cfg[\"epochs\"]))\n    ckpt_dir = logs_dir(config) / \"checkpoints\"\n    ckpt_dir.mkdir(parents=True, exist_ok=True)\n    ckpt_path = ckpt_dir / f\"{mode}_{adapter_kind}_best.pt\"\n    latest_ckpt_path = ckpt_dir / f\"{mode}_{adapter_kind}_latest.pt\"\n    history_path = logs_dir(config) / f\"train_history_{adapter_kind}_{mode}.json\"\n    start = time.time()\n    start_epoch = 0\n    if bool(training_cfg.get(\"auto_resume\", True)) and latest_ckpt_path.exists():\n        payload = torch.load(latest_ckpt_path, map_location=device)\n        adapter.load_state_dict(payload[\"state_dict\"])\n        optimizer.load_state_dict(payload[\"optimizer_state\"])\n        scheduler.load_state_dict(payload[\"scheduler_state\"])\n        history = list(payload.get(\"history\", []))\n        best_val = float(payload.get(\"best_val_loss\", float(\"inf\")))\n        start_epoch = int(payload.get(\"epoch\", 0))\n        start = time.time() - float(payload.get(\"elapsed_s\", 0.0))\n        epochs_without_improvement = int(payload.get(\"epochs_without_improvement\", 0))\n        print(\n            f\"[adapter-train] auto-resume from {latest_ckpt_path} starting at epoch {start_epoch + 1}/{int(training_cfg['epochs'])}\",\n            flush=True,\n        )\n    for epoch in range(start_epoch, int(training_cfg[\"epochs\"])):\n        print(f\"[adapter-train] epoch {epoch + 1}/{int(training_cfg['epochs'])} started\", flush=True)\n        adapter.train()\n        train_losses: list[float] = []\n        num_train_batches = len(train_loader)\n        for batch_idx, (x, y) in enumerate(train_loader, start=1):\n            x = x.to(device=device, dtype=torch.float32)\n            y = y.to(device=device, dtype=torch.float32)\n            optimizer.zero_grad(set_to_none=True)\n            loss = _adapter_loss_fn(adapter, x, y, residual_weight)\n            loss.backward()\n            optimizer.step()\n            train_losses.append(float(loss.detach().cpu()))\n            if batch_idx == 1 or batch_idx % 100 == 0 or batch_idx == num_train_batches:\n                print(\n                    f\"[adapter-train] epoch {epoch + 1} batch {batch_idx}/{num_train_batches} loss={train_losses[-1]:.6f}\",\n                    flush=True,\n                )\n        adapter.eval()\n        val_losses: list[float] = []\n        num_val_batches = len(val_loader)\n        with torch.no_grad():\n            for batch_idx, (x, y) in enumerate(val_loader, start=1):\n                x = x.to(device=device, dtype=torch.float32)\n                y = y.to(device=device, dtype=torch.float32)\n                val_losses.append(float(_adapter_loss_fn(adapter, x, y, residual_weight).detach().cpu()))\n                if batch_idx == 1 or batch_idx % 100 == 0 or batch_idx == num_val_batches:\n                    print(\n                        f\"[adapter-train] epoch {epoch + 1} val_batch {batch_idx}/{num_val_batches} loss={val_losses[-1]:.6f}\",\n                        flush=True,\n                    )\n        train_loss = float(np.mean(train_losses))\n        val_loss = float(np.mean(val_losses))\n        scheduler.step(val_loss)\n        improved = val_loss < best_val\n        if improved:\n            best_val = val_loss\n            epochs_without_improvement = 0\n        else:\n            epochs_without_improvement += 1\n        record = {\n            \"epoch\": epoch + 1,\n            \"train_loss\": train_loss,\n            \"val_loss\": val_loss,\n            \"lr\": float(optimizer.param_groups[0][\"lr\"]),\n            \"elapsed_s\": float(time.time() - start),\n        }\n        history.append(record)\n        payload = {\n            \"epoch\": epoch + 1,\n            \"adapter_kind\": str(adapter_kind),\n            \"state_dict\": adapter.state_dict(),\n            \"optimizer_state\": optimizer.state_dict(),\n            \"scheduler_state\": scheduler.state_dict(),\n            \"history\": history,\n            \"best_val_loss\": best_val,\n            \"elapsed_s\": record[\"elapsed_s\"],\n            \"epochs_without_improvement\": epochs_without_improvement,\n        }\n        torch.save(payload, latest_ckpt_path)\n        print(\n            f\"[adapter-train] epoch {epoch + 1} done train_loss={train_loss:.6f} val_loss={val_loss:.6f} \"\n            f\"lr={record['lr']:.6e} elapsed_s={record['elapsed_s']:.1f}\",\n            flush=True,\n        )\n        print(f\"[adapter-train] latest checkpoint saved to {latest_ckpt_path}\", flush=True)\n        if bool(training_cfg.get(\"save_epoch_checkpoints\", True)):\n            epoch_ckpt_path = ckpt_dir / f\"{mode}_{adapter_kind}_epoch_{epoch + 1:03d}.pt\"\n            torch.save(payload, epoch_ckpt_path)\n            print(f\"[adapter-train] epoch checkpoint saved to {epoch_ckpt_path}\", flush=True)\n        if improved:\n            torch.save(payload, ckpt_path)\n            print(f\"[adapter-train] new best checkpoint saved to {ckpt_path} with val_loss={best_val:.6f}\", flush=True)\n        if epochs_without_improvement >= early_stop_patience:\n            print(\n                f\"[adapter-train] early stop at epoch {epoch + 1} after {epochs_without_improvement} non-improving epochs\",\n                flush=True,\n            )\n            break\n    save_json(history_path, {\"history\": history, \"best_val_loss\": best_val, \"adapter_kind\": str(adapter_kind)})\n    print(f\"[adapter-train] finished best_val_loss={best_val:.6f} history={history_path}\", flush=True)\n    return ckpt_path\n\n\ndef load_adapter_checkpoint(config: SystemConfig, checkpoint_path: Path | None = None, adapter_kind: str | None = None) -> ResidualAdapter:\n    if checkpoint_path is None:\n        if adapter_kind is None:\n            adapter_kind = str(config.raw.get(\"adapter\", {}).get(\"default_kind\", \"fb_lara\"))\n        checkpoint_path = logs_dir(config) / \"checkpoints\" / f\"full_{adapter_kind}_best.pt\"\n    device = resolve_torch_device(config)\n    payload = torch.load(checkpoint_path, map_location=device)\n    resolved_kind = str(adapter_kind or payload.get(\"adapter_kind\", \"fb_lara\"))\n    model = instantiate_adapter(resolved_kind).to(device=device, dtype=torch.float32)\n    model.load_state_dict(payload[\"state_dict\"])\n    model.eval()\n    print(f\"[adapter-train] loaded {resolved_kind} checkpoint {checkpoint_path} on device={device}\", flush=True)\n    return model\n",
  "src/zakotfs_novelty/utils.py": "from __future__ import annotations\n\nimport json\nimport math\nimport random\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nfrom statistics import NormalDist\n\nfrom .params import SystemConfig\n\n\ndef set_global_seed(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef resolve_torch_device(config: SystemConfig | None = None, explicit: str | None = None) -> torch.device:\n    requested = (explicit if explicit is not None else str(config.raw.get(\"device\", \"auto\")) if config is not None else \"auto\").lower()\n    if requested == \"cuda\":\n        if not torch.cuda.is_available():\n            raise RuntimeError(\"Requested device=cuda but torch.cuda.is_available() is False\")\n        return torch.device(\"cuda\")\n    if requested == \"mps\":\n        if not hasattr(torch.backends, \"mps\") or not torch.backends.mps.is_available():\n            raise RuntimeError(\"Requested device=mps but torch.backends.mps.is_available() is False\")\n        return torch.device(\"mps\")\n    if requested == \"cpu\":\n        return torch.device(\"cpu\")\n    if requested != \"auto\":\n        return torch.device(requested)\n    if torch.cuda.is_available():\n        return torch.device(\"cuda\")\n    if hasattr(torch.backends, \"mps\") and torch.backends.mps.is_available():\n        return torch.device(\"mps\")\n    return torch.device(\"cpu\")\n\n\ndef pin_memory_enabled(device: torch.device) -> bool:\n    return device.type == \"cuda\"\n\n\ndef repo_root(config: SystemConfig) -> Path:\n    return config.root\n\n\ndef ensure_dir(path: Path) -> Path:\n    path.mkdir(parents=True, exist_ok=True)\n    return path\n\n\ndef results_dir(config: SystemConfig) -> Path:\n    return ensure_dir(repo_root(config) / config.raw[\"paths\"][\"results_dir\"])\n\n\ndef logs_dir(config: SystemConfig) -> Path:\n    return ensure_dir(repo_root(config) / config.raw[\"paths\"][\"logs_dir\"])\n\n\ndef report_dir(config: SystemConfig) -> Path:\n    return ensure_dir(repo_root(config) / config.raw[\"paths\"][\"report_dir\"])\n\n\ndef centered_coord_to_index(coord: int, size: int) -> int:\n    return coord % size\n\n\ndef index_to_centered_coord(index: int, size: int) -> int:\n    half = size // 2\n    return index if index <= half else index - size\n\n\ndef centered_mesh(config: SystemConfig) -> tuple[np.ndarray, np.ndarray]:\n    k = np.array([index_to_centered_coord(i, config.M) for i in range(config.M)], dtype=int)\n    l = np.array([index_to_centered_coord(i, config.N) for i in range(config.N)], dtype=int)\n    return np.meshgrid(k, l, indexing=\"ij\")\n\n\ndef complex_noise(shape: tuple[int, ...], variance: float, rng: np.random.Generator) -> np.ndarray:\n    scale = math.sqrt(variance / 2.0)\n    return scale * (rng.standard_normal(shape) + 1j * rng.standard_normal(shape))\n\n\ndef save_json(path: Path, payload: dict[str, Any]) -> None:\n    with path.open(\"w\", encoding=\"utf-8\") as handle:\n        json.dump(payload, handle, indent=2, sort_keys=True)\n\n\ndef load_json(path: Path) -> dict[str, Any]:\n    with path.open(\"r\", encoding=\"utf-8\") as handle:\n        return json.load(handle)\n\n\ndef bootstrap_mean_ci(values: np.ndarray, seed: int, num_bootstrap: int = 400, alpha: float = 0.05) -> tuple[float, float]:\n    rng = np.random.default_rng(seed)\n    if values.size == 1:\n        scalar = float(values[0])\n        return scalar, scalar\n    means = []\n    for _ in range(num_bootstrap):\n        sample = rng.choice(values, size=values.size, replace=True)\n        means.append(float(np.mean(sample)))\n    means = np.sort(np.array(means))\n    lo = float(np.quantile(means, alpha / 2.0))\n    hi = float(np.quantile(means, 1.0 - alpha / 2.0))\n    return lo, hi\n\n\ndef snr_to_noise_variance(E_signal: float, snr_db: float, num_samples: int) -> float:\n    snr_lin = 10.0 ** (snr_db / 10.0)\n    n0 = E_signal / (num_samples * snr_lin)\n    return float(n0)\n\n\ndef wilson_ci(num_errors: int, num_bits: int, confidence: float = 0.95) -> tuple[float, float]:\n    if num_bits <= 0:\n        return 0.0, 0.0\n    if num_errors < 0 or num_errors > num_bits:\n        raise ValueError(\"num_errors must satisfy 0 <= num_errors <= num_bits\")\n    z = NormalDist().inv_cdf(0.5 + confidence / 2.0)\n    phat = num_errors / num_bits\n    denom = 1.0 + (z**2) / num_bits\n    center = (phat + (z**2) / (2.0 * num_bits)) / denom\n    radius = (z / denom) * math.sqrt((phat * (1.0 - phat) / num_bits) + (z**2) / (4.0 * num_bits**2))\n    return max(0.0, center - radius), min(1.0, center + radius)\n",
  "src/zakotfs_novelty/waveform.py": "from __future__ import annotations\n\nimport numpy as np\nfrom functools import lru_cache\n\nfrom .modulation import sample_symbols\nfrom .params import SystemConfig\nfrom .pulses import chirp_spreading_filter, mn_periodic_chirp_filter\n\n\ndef data_symbols(modulation: str, config: SystemConfig, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:\n    return sample_symbols(modulation, (config.M, config.N), rng)\n\n\ndef data_waveform(symbols: np.ndarray) -> np.ndarray:\n    return (symbols / np.sqrt(symbols.size)).astype(np.complex64)\n\n\ndef point_pilot(config: SystemConfig) -> np.ndarray:\n    pilot = np.zeros((config.M, config.N), dtype=np.complex64)\n    pilot[int(config.frame[\"pilot_delay_bin\"]), int(config.frame[\"pilot_doppler_bin\"])] = 1.0 + 0.0j\n    return pilot\n\n\ndef periodic_twisted_convolution(a: np.ndarray, b: np.ndarray) -> np.ndarray:\n    M, N = a.shape\n    out = np.zeros((M, N), dtype=np.complex64)\n    Q = M * N\n    for k in range(M):\n        for l in range(N):\n            acc = 0.0j\n            for kp in range(M):\n                for lp in range(N):\n                    acc += a[(k - kp) % M, (l - lp) % N] * b[kp, lp] * np.exp(1j * 2 * np.pi * kp * (l - lp) / Q)\n            out[k, l] = acc\n    return out\n\n\ndef spread_pilot(config: SystemConfig) -> np.ndarray:\n    kp = int(config.frame[\"pilot_delay_bin\"])\n    lp = int(config.frame[\"pilot_doppler_bin\"])\n    return _spread_pilot_exact(config.M, config.N, config.q, kp, lp).copy()\n\n\n@lru_cache(maxsize=8)\ndef _spread_pilot_exact(M: int, N: int, q: int, kp: int, lp: int) -> np.ndarray:\n    Q = M * N\n    # Eq. (2) uses the MN-periodic extension of the chirp filter, not the MxN crop.\n    w = np.exp(1j * 2 * np.pi * q * ((np.arange(Q)[:, None] ** 2) + (np.arange(Q)[None, :] ** 2)) / Q) / Q\n    xs = np.zeros((M, N), dtype=np.complex64)\n    if kp == 0 and lp == 0:\n        phase_n = np.exp(1j * 2 * np.pi * (np.arange(N)[:, None] * np.arange(N)[None, :]) / N)\n        for k in range(M):\n            for l in range(N):\n                acc = 0.0j\n                for n in range(N):\n                    inner = 0.0j\n                    for m in range(M):\n                        inner += w[(k - n * M) % Q, (l - m * N) % Q]\n                    acc += phase_n[n, l] * inner\n                xs[k, l] = acc\n        return xs\n    for k in range(M):\n        for l in range(N):\n            acc = 0.0j\n            for n in range(N):\n                for m in range(M):\n                    phase = np.exp(1j * 2 * np.pi * lp * n / N) * np.exp(\n                        1j * 2 * np.pi * (l - lp - m * N) * (kp + n * M) / Q\n                    )\n                    acc += w[(k - kp - n * M) % Q, (l - lp - m * N) % Q] * phase\n            xs[k, l] = acc\n    return xs\n\n\ndef superimposed_frame(data_dd: np.ndarray, spread_dd: np.ndarray, E_d: float, E_p: float) -> np.ndarray:\n    return (np.sqrt(E_d) * data_dd + np.sqrt(E_p) * spread_dd).astype(np.complex64)\n",
  "configs/adapter_eval.yaml": "inherits: system.yaml\n\nevaluation:\n  nmse_vs_pdr:\n    modulation: bpsk\n    data_snr_db: 15.0\n    pdr_db: [-5.0, 0.0, 5.0, 10.0, 15.0, 20.0, 25.0]\n    realizations: 400\n    methods: [conventional, cnn, generic, fb_lara]\n  nmse_vs_snr:\n    modulation: bpsk\n    pdr_db: 5.0\n    data_snr_db: [0.0, 3.0, 6.0, 9.0, 12.0, 15.0, 18.0, 21.0, 24.0]\n    realizations: 400\n    methods: [conventional, cnn, generic, fb_lara]\n  ber_vs_pdr:\n    modulation: bpsk\n    data_snr_db: 18.0\n    pdr_db: [-5.0, 0.0, 5.0, 10.0, 15.0, 20.0, 25.0]\n    target_bit_errors: 200\n    max_bits: 1000000\n    min_frames: 20\n    solver: dense_torch\n    methods: [conventional, cnn, generic, fb_lara, perfect]\n  ber_vs_snr:\n    pdr_db: 5.0\n    data_snr_db: [0.0, 3.0, 6.0, 9.0, 12.0, 15.0, 18.0, 21.0, 24.0]\n    modulation: [bpsk, 8qam_cross]\n    target_bit_errors: 200\n    max_bits: 1000000\n    min_frames: 20\n    solver: dense_torch\n    methods: [conventional, cnn, generic, fb_lara, perfect]\n\nsmoke:\n  evaluation:\n    nmse_vs_pdr:\n      realizations: 2\n    nmse_vs_snr:\n      realizations: 2\n    ber_vs_pdr:\n      target_bit_errors: 5\n      max_bits: 5000\n      min_frames: 2\n    ber_vs_snr:\n      target_bit_errors: 5\n      max_bits: 5000\n      min_frames: 2\n",
  "configs/adapter_eval_mps.yaml": "inherits: adapter_eval.yaml\n\ndevice: mps\n",
  "configs/adapter_train.yaml": "inherits: system.yaml\n\nadapter_dataset:\n  train_size_total: 48000\n  val_size_total: 12000\n  training_snr_db: 15.0\n  training_pdr_db: [0.0, 5.0, 20.0, 25.0, 30.0, 35.0]\n  modulation: bpsk\n  num_workers: 0\n  baseline_manifest_paths:\n    train: ../results/datasets/train_full.json\n    val: ../results/datasets/val_full.json\n\nadapter_training:\n  batch_size: 128\n  epochs: 8\n  learning_rate: 0.001\n  residual_penalty: 0.001\n  auto_resume: true\n  save_epoch_checkpoints: true\n  early_stop_patience: 3\n  scheduler:\n    factor: 0.8\n    patience: 1\n    min_lr: 1.0e-06\n\nsmoke:\n  adapter_dataset:\n    train_size_total: 120\n    val_size_total: 24\n  adapter_training:\n    batch_size: 16\n    epochs: 1\n    early_stop_patience: 1\n",
  "configs/adapter_train_mps.yaml": "inherits: adapter_train.yaml\n\ndevice: mps\n",
  "configs/system.yaml": "seed: 12345\ndevice: auto\ndtype: complex64\n\nframe:\n  nu_p_hz: 30000.0\n  tau_p_s: 3.3333333333333335e-05\n  M: 31\n  N: 37\n  q: 3\n  T_s: 0.0012333333333333332\n  B_hz: 930000.0\n  pilot_delay_bin: 0\n  pilot_doppler_bin: 0\n\npulse:\n  kind: gs\n  alpha_tau: 0.044\n  alpha_nu: 0.044\n  Omega_tau: 1.0278\n  Omega_nu: 1.0278\n  no_expansion: true\n\nchannel:\n  model: vehicular_a\n  num_paths: 6\n  max_doppler_hz: 815.0\n  max_delay_s: 2.51e-06\n  delays_s: [0.0, 3.1e-07, 7.1e-07, 1.09e-06, 1.73e-06, 2.51e-06]\n  relative_powers_db: [0.0, -1.0, -9.0, -10.0, -15.0, -20.0]\n  summation_range: [-1, 0, 1]\n  gain_distribution: complex_gaussian\n  gain_normalization: normalize_to_unit_average_power\n\nestimation:\n  support_center: [0, 0]\n  threshold_multiplier: 3.0\n  thresholding: both\n  use_thresholded_for_detection: true\n  derive_support_from_lattice: true\n\ndetection:\n  method: mmse\n  cg_tol: 1.0e-8\n  cg_maxiter: 200\n\nnumerics:\n  effective_channel_method: fast\n  pulse_quadrature:\n    delay_points: 2049\n    doppler_points: 2049\n    delay_span: 10.0\n    doppler_span: 10.0\n\npaths:\n  results_dir: results\n  logs_dir: logs\n  report_dir: report\n\nbackbone:\n  checkpoint_path: ../logs/checkpoints/full_cnn_best.pt\n\nadapter:\n  default_kind: fb_lara\n  checkpoint_paths:\n    generic: logs/checkpoints/full_generic_best.pt\n    fb_lara: logs/checkpoints/full_fb_lara_best.pt\n",
  "configs/train.yaml": "inherits: system.yaml\n\ndataset:\n  train_size_total: 480000\n  val_size_total: 120000\n  train_size_per_pdr: 80000\n  val_size_per_pdr: 20000\n  training_snr_db: 15.0\n  training_pdr_db: [0.0, 5.0, 20.0, 25.0, 30.0, 35.0]\n  mode: full\n  save_npz: true\n  num_workers: 0\n  materialize_batch_size: 128\n  materialize_num_workers: 0\n  cache_in_ram: false\n\ntraining:\n  modulation: bpsk\n  batch_size: 64\n  epochs: 50\n  learning_rate: 0.0005\n  auto_resume: true\n  save_epoch_checkpoints: true\n  optimizer: adam\n  scheduler:\n    kind: ReduceLROnPlateau\n    factor: 0.8\n    patience: 3\n    min_lr: 1.0e-06\n  save_best_only: true\n  checkpoint_name: cnn_best.pt\n\nsmoke:\n  dataset:\n    train_size_total: 24\n    val_size_total: 12\n    train_size_per_pdr: 4\n    val_size_per_pdr: 2\n  training:\n    epochs: 1\n    batch_size: 4\n",
  "tests/test_adapter_dataset.py": "from __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom zakotfs_novelty.adapter import AdapterFeatureDataset\n\n\ndef test_adapter_dataset_slices_generic_channels(tmp_path: Path) -> None:\n    inputs = np.arange(8 * 2 * 2, dtype=np.float32).reshape(1, 8, 2, 2)\n    targets = np.ones((1, 2, 2, 2), dtype=np.float32)\n    inputs_path = tmp_path / \"inputs.npy\"\n    targets_path = tmp_path / \"targets.npy\"\n    pdr_path = tmp_path / \"pdr.npy\"\n    np.save(inputs_path, inputs)\n    np.save(targets_path, targets)\n    np.save(pdr_path, np.array([5.0], dtype=np.float32))\n    manifest = {\n        \"size\": 1,\n        \"inputs_path\": inputs_path.name,\n        \"targets_path\": targets_path.name,\n        \"pdr_labels_path\": pdr_path.name,\n    }\n    manifest_path = tmp_path / \"dataset.json\"\n    manifest_path.write_text(json.dumps(manifest), encoding=\"utf-8\")\n    generic = AdapterFeatureDataset(manifest_path, adapter_kind=\"generic\")\n    fb_lara = AdapterFeatureDataset(manifest_path, adapter_kind=\"fb_lara\")\n    x_generic, y_generic = generic[0]\n    x_fb_lara, y_fb_lara = fb_lara[0]\n    assert x_generic.shape == (6, 2, 2)\n    assert x_fb_lara.shape == (8, 2, 2)\n    assert np.allclose(x_generic.numpy()[0], inputs[0, 0])\n    assert np.allclose(x_generic.numpy()[4], inputs[0, 6])\n    assert np.allclose(x_generic.numpy()[5], inputs[0, 7])\n    assert np.allclose(y_generic.numpy(), targets[0])\n    assert np.allclose(y_fb_lara.numpy(), targets[0])\n",
  "tests/test_adapter_features.py": "from __future__ import annotations\n\nimport numpy as np\n\nfrom zakotfs_novelty.adapter import build_fb_lara_features, feature_channel_indices, residual_target\n\n\ndef test_fb_lara_feature_stack_layout() -> None:\n    h_raw = np.array([[1 + 2j, 3 + 4j]], dtype=np.complex64)\n    h_base = np.array([[0.5 + 1.5j, 2.5 + 3.5j]], dtype=np.complex64)\n    h_alias = np.array([[0.1 - 0.2j, -0.3 + 0.4j]], dtype=np.complex64)\n    features = build_fb_lara_features(h_raw, h_base, h_alias)\n    target = residual_target(h_raw, h_base)\n    assert features.shape == (8, 1, 2)\n    assert np.allclose(features[0], [[1.0, 3.0]])\n    assert np.allclose(features[1], [[2.0, 4.0]])\n    assert np.allclose(features[4], [[0.1, -0.3]])\n    assert np.allclose(features[5], [[-0.2, 0.4]])\n    assert np.allclose(features[6], [[0.5, 0.5]])\n    assert np.allclose(features[7], [[0.5, 0.5]])\n    assert target.shape == (2, 1, 2)\n    assert np.allclose(target[0], [[0.5, 0.5]])\n    assert np.allclose(target[1], [[0.5, 0.5]])\n    assert feature_channel_indices(\"generic\") == (0, 1, 2, 3, 6, 7)\n    assert feature_channel_indices(\"fb_lara\") == tuple(range(8))\n",
  "tests/test_adapter_models.py": "from __future__ import annotations\n\nimport torch\n\nfrom zakotfs_novelty.cnn_model import GenericResidualAdapter, LatticeAliasAdapter\n\n\ndef test_fb_lara_zero_init_outputs_zero_residual() -> None:\n    model = LatticeAliasAdapter()\n    x = torch.randn(3, 8, 27, 43, dtype=torch.float32)\n    y = model(x)\n    assert y.shape == (3, 2, 27, 43)\n    assert torch.count_nonzero(y).item() == 0\n    assert model.num_parameters == 6594\n\n\ndef test_generic_adapter_zero_init_outputs_zero_residual() -> None:\n    model = GenericResidualAdapter()\n    x = torch.randn(2, 6, 27, 43, dtype=torch.float32)\n    y = model(x)\n    assert y.shape == (2, 2, 27, 43)\n    assert torch.count_nonzero(y).item() == 0\n",
  "tests/test_device_resolution.py": "from __future__ import annotations\n\nfrom zakotfs_novelty.utils import resolve_torch_device\n\n\ndef test_resolve_torch_device_cpu_explicit() -> None:\n    assert resolve_torch_device(explicit=\"cpu\").type == \"cpu\"\n",
  "pytest.ini": "[pytest]\npythonpath = src\ntestpaths = tests\n"
}

for rel_path, text in FILE_PAYLOADS.items():
    out_path = WORK_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(text, encoding='utf-8')

if str(WORK_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(WORK_ROOT / 'src'))

print(f'[setup] wrote {len(FILE_PAYLOADS)} embedded files under {WORK_ROOT}')


In [ ]:
from zakotfs_novelty.params import load_config

def apply_runtime_overrides(cfg, *, train_cfg: bool):
    cfg.raw['device'] = DEVICE_OVERRIDE
    cfg.raw.setdefault('backbone', {})['checkpoint_path'] = str(BASELINE_CHECKPOINT)
    cfg.raw['paths'] = {
        'results_dir': str((WORK_ROOT / 'results').resolve()),
        'logs_dir': str((WORK_ROOT / 'logs').resolve()),
        'report_dir': str((WORK_ROOT / 'report').resolve()),
    }
    if train_cfg:
        cfg.raw['adapter_dataset']['train_size_total'] = int(ADAPTER_TRAIN_SIZE)
        cfg.raw['adapter_dataset']['val_size_total'] = int(ADAPTER_VAL_SIZE)
        if BASELINE_TRAIN_MANIFEST is not None and BASELINE_VAL_MANIFEST is not None:
            cfg.raw['adapter_dataset']['baseline_manifest_paths'] = {
                'train': str(BASELINE_TRAIN_MANIFEST),
                'val': str(BASELINE_VAL_MANIFEST),
            }
        else:
            cfg.raw['adapter_dataset']['baseline_manifest_paths'] = {}
    return cfg

train_cfg = apply_runtime_overrides(load_config(WORK_ROOT / 'configs' / 'adapter_train.yaml'), train_cfg=True)
eval_cfg = apply_runtime_overrides(load_config(WORK_ROOT / 'configs' / 'adapter_eval.yaml'), train_cfg=False)

print('[setup] runtime train config ready')
print(train_cfg.raw['adapter_dataset'])
print('[setup] runtime eval device:', eval_cfg.raw['device'])


In [ ]:
if RUN_TESTS:
    env = dict(os.environ)
    env['PYTHONPATH'] = str(WORK_ROOT / 'src')
    subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests', '--rootdir', str(WORK_ROOT), '-c', str(WORK_ROOT / 'pytest.ini')], cwd=WORK_ROOT, env=env, check=True)
else:
    print('[tests] skipped')


In [ ]:
from zakotfs_novelty.adapter import generate_adapter_dataset

train_adapter_manifest = WORK_ROOT / 'results' / 'adapter_datasets' / 'train_adapter.json'
val_adapter_manifest = WORK_ROOT / 'results' / 'adapter_datasets' / 'val_adapter.json'

if RUN_DATASET_BUILD:
    train_adapter_manifest = generate_adapter_dataset(train_cfg, 'train', force=False)
    val_adapter_manifest = generate_adapter_dataset(train_cfg, 'val', force=False)

print({'train_adapter_manifest': str(train_adapter_manifest), 'val_adapter_manifest': str(val_adapter_manifest)})


In [ ]:
from pathlib import Path
from zakotfs_novelty.training import train_adapter

adapter_ckpts = {}
if RUN_TRAIN_GENERIC:
    adapter_ckpts['generic'] = train_adapter(train_cfg, Path(train_adapter_manifest), Path(val_adapter_manifest), adapter_kind='generic', mode='full')
if RUN_TRAIN_FB_LARA:
    adapter_ckpts['fb_lara'] = train_adapter(train_cfg, Path(train_adapter_manifest), Path(val_adapter_manifest), adapter_kind='fb_lara', mode='full')

print({key: str(value) for key, value in adapter_ckpts.items()})


In [ ]:
from pathlib import Path
from zakotfs_novelty.adapter import load_frozen_backbone
from zakotfs_novelty.evaluation import evaluate_ber_vs_pdr, evaluate_ber_vs_snr, evaluate_nmse_vs_pdr, evaluate_nmse_vs_snr, save_eval_outputs
from zakotfs_novelty.training import load_adapter_checkpoint

def apply_available_methods(cfg, adapters):
    available = ['conventional', 'cnn'] + sorted(adapters.keys())
    available_with_perfect = available + ['perfect']
    for section in ['nmse_vs_pdr', 'nmse_vs_snr']:
        cfg.raw['evaluation'][section]['methods'] = [method for method in cfg.raw['evaluation'][section]['methods'] if method in available]
    for section in ['ber_vs_pdr', 'ber_vs_snr']:
        cfg.raw['evaluation'][section]['methods'] = [method for method in cfg.raw['evaluation'][section]['methods'] if method in available_with_perfect]

backbone, backbone_device = load_frozen_backbone(eval_cfg)
loaded_adapters = {}
for kind in ['generic', 'fb_lara']:
    if kind in adapter_ckpts:
        loaded_adapters[kind] = load_adapter_checkpoint(eval_cfg, Path(adapter_ckpts[kind]), adapter_kind=kind)
    else:
        ckpt_path = WORK_ROOT / 'logs' / 'checkpoints' / f'full_{kind}_best.pt'
        if ckpt_path.exists():
            loaded_adapters[kind] = load_adapter_checkpoint(eval_cfg, ckpt_path, adapter_kind=kind)

apply_available_methods(eval_cfg, loaded_adapters)
print({'backbone_device': str(backbone_device), 'adapters': list(loaded_adapters.keys())})


In [ ]:
if RUN_EVAL_NMSE:
    nmse_pdr = evaluate_nmse_vs_pdr(eval_cfg, backbone, loaded_adapters, mode='full')
    save_eval_outputs(nmse_pdr, 'nmse', 'adapter_nmse_vs_pdr_full', eval_cfg)
    nmse_snr = evaluate_nmse_vs_snr(eval_cfg, backbone, loaded_adapters, mode='full')
    save_eval_outputs(nmse_snr, 'nmse', 'adapter_nmse_vs_snr_full', eval_cfg)
else:
    print('[eval] NMSE skipped')


In [ ]:
if RUN_EVAL_BER:
    ber_pdr = evaluate_ber_vs_pdr(eval_cfg, backbone, loaded_adapters, mode='full')
    save_eval_outputs(ber_pdr, 'ber', 'adapter_ber_vs_pdr_full', eval_cfg)
    ber_snr = evaluate_ber_vs_snr(eval_cfg, backbone, loaded_adapters, mode='full')
    save_eval_outputs(ber_snr, 'ber', 'adapter_ber_vs_snr_full', eval_cfg)
else:
    print('[eval] BER skipped')


In [ ]:
results_dir = WORK_ROOT / 'results'
logs_dir = WORK_ROOT / 'logs'
print('[artifacts] results')
for path in sorted(results_dir.rglob('*')):
    if path.is_file():
        print(path)
print('[artifacts] logs')
for path in sorted(logs_dir.rglob('*')):
    if path.is_file():
        print(path)
